# EXOLENE | Oil-spill analysis notebook

This notebook contains three related but distinct workstreams: **pixel-level oil-spill segmentation**, **image-level CSIRO Oil/No-Oil classification**, and **Sentinel-1 SAFE product exploration**. The two ML tasks use separate datasets, labels, models, checkpoints and output folders.

The notebook preserves the exploratory experiments and pipeline investigations. It is not a claim that every workflow has been rerun successfully in this environment.

## Table of contents

1. Environment, imports and portable configuration
2. Initial segmentation experiments and U-Net
3. CSIRO image-level classification
4. Saved segmentation-model inference example
5. Sentinel-1 SAFE, XML, VV/VH and raster exploration
6. Segmentation dataset inventory and pipeline inspection
7. Validation, prediction and mask-quality audits
8. Training-script and configuration investigations
9. Metrics, checkpoints and weighted CE + Dice experiments
10. Checkpoint comparisons, threshold analysis and test evaluation
11. Results interpretation, limitations and next steps

**Execution note:** Original cell order is retained to avoid silently breaking notebook state dependencies. Some experiments were originally run out of order (as shown by their saved execution counts); see the audit and cell mapping.

## 1. Environment, imports and portable configuration

Central configuration defines the project root, independent segmentation/classification dataset roots, reproducibility seed, compute device and separate output directories. Update the dataset paths here or set the documented environment variables before running data-dependent cells.

In [ ]:
from pathlib import Path
import os
import random
import re
import zipfile
from io import BytesIO

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix
)

# ---------------------------------------------------------------------
# Project configuration
# ---------------------------------------------------------------------
# Run the notebook with the project folder as the working directory, or
# set EXOLENE_PROJECT_ROOT / dataset-specific environment variables.
# These defaults are project-relative; edit them if your folders differ.
PROJECT_ROOT = Path(os.environ.get("EXOLENE_PROJECT_ROOT", Path.cwd())).expanduser()

SEG_DATASET_DIR = Path(os.environ.get(
    "EXOLENE_SEG_DATASET_DIR",
    PROJECT_ROOT / "oilspill-detection-dataset_20260912-1b9a62c"
)).expanduser()

CLS_DATASET_DIR = Path(os.environ.get(
    "EXOLENE_CLS_DATASET_DIR",
    PROJECT_ROOT / "2022-12-15_Blondeau-Patissier_David_57430v1"
)).expanduser()

# Keep artifacts from the two independent ML tasks separate.
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
SEG_OUTPUT_DIR = OUTPUT_ROOT / "segmentation"
CLS_OUTPUT_DIR = OUTPUT_ROOT / "classification"
SEG_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CLS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Project root:", PROJECT_ROOT.resolve())
print("Segmentation dataset:", SEG_DATASET_DIR.resolve(), "| exists:", SEG_DATASET_DIR.exists())
print("Classification dataset:", CLS_DATASET_DIR.resolve(), "| exists:", CLS_DATASET_DIR.exists())
print("Segmentation outputs:", SEG_OUTPUT_DIR.resolve())
print("Classification outputs:", CLS_OUTPUT_DIR.resolve())
print("Device:", DEVICE)

# Missing inputs are reported here; the relevant workflow cells also check
# the exact files they need before loading them.


## 2. Initial segmentation experiments and U-Net

This section explores paired image/mask data, model architecture, losses, training and test visualisation. The early experiments are retained as historical experiments; later pipeline-based segmentation work appears in the following sections.

In [ ]:
SEG_IMAGE_DIR = SEG_DATASET_DIR / "IMAGES" / "IMG-RGB"
SEG_MASK_DIR = SEG_DATASET_DIR / "IMAGES" / "LABELS-1D"
SEG_SPLIT_DIR = SEG_DATASET_DIR / "splits"

print("Image directory:", SEG_IMAGE_DIR)
print("Mask directory:", SEG_MASK_DIR)
print("Split directory:", SEG_SPLIT_DIR)

if not SEG_DATASET_DIR.exists():
    raise FileNotFoundError(f"Segmentation dataset directory not found: {SEG_DATASET_DIR}")

assert SEG_IMAGE_DIR.exists()
assert SEG_MASK_DIR.exists()
assert (SEG_SPLIT_DIR / "train.csv").exists()
assert (SEG_SPLIT_DIR / "val.csv").exists()
assert (SEG_SPLIT_DIR / "test.csv").exists()

print("✓ Segmentation paths verified.")


In [ ]:
def read_split_csv(path):
    df = pd.read_csv(path)
    print(path.name, "columns:", list(df.columns))
    print("rows:", len(df))
    return df

train_seg_df = read_split_csv(SEG_SPLIT_DIR / "train.csv")
val_seg_df = read_split_csv(SEG_SPLIT_DIR / "val.csv")
test_seg_df = read_split_csv(SEG_SPLIT_DIR / "test.csv")


In [ ]:
# Inspect the actual split-column names before constructing paths.
print(train_seg_df.head())


In [ ]:
def find_image_column(df):
    candidates = ["image_path", "image", "img", "filename", "file"]
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"No image column found. Available columns: {list(df.columns)}")

def find_mask_column(df):
    candidates = ["mask_path", "mask", "label_path", "label", "mask_file"]
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"No mask column found. Available columns: {list(df.columns)}")

SEG_IMAGE_COL = find_image_column(train_seg_df)
SEG_MASK_COL = find_mask_column(train_seg_df)

print("Image column:", SEG_IMAGE_COL)
print("Mask column:", SEG_MASK_COL)


In [ ]:
def resolve_dataset_path(root, value):
    value = Path(str(value))
    candidates = [
        root / value,
        root / value.name,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate

    # Try common image/mask directories if the CSV stores only filenames.
    for base in [SEG_IMAGE_DIR, SEG_MASK_DIR]:
        candidate = base / value.name
        if candidate.exists():
            return candidate

    raise FileNotFoundError(
        f"Could not resolve dataset file '{value}'. Checked relative paths and "
        f"the image/mask directories."
    )

sample_row = train_seg_df.iloc[0]
sample_image_path = resolve_dataset_path(SEG_DATASET_DIR, sample_row[SEG_IMAGE_COL])
sample_mask_path = resolve_dataset_path(SEG_DATASET_DIR, sample_row[SEG_MASK_COL])

print("Sample image:", sample_image_path)
print("Sample mask:", sample_mask_path)

sample_image = Image.open(sample_image_path)
sample_mask = Image.open(sample_mask_path)

print("Image size/mode:", sample_image.size, sample_image.mode)
print("Mask size/mode:", sample_mask.size, sample_mask.mode)
print("Mask values:", np.unique(np.asarray(sample_mask)))


In [ ]:
class SegmentationDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]

        image_path = resolve_dataset_path(SEG_DATASET_DIR, row[SEG_IMAGE_COL])
        mask_path = resolve_dataset_path(SEG_DATASET_DIR, row[SEG_MASK_COL])

        image = np.asarray(
            Image.open(image_path).convert("RGB"),
            dtype=np.float32
        )
        mask = np.asarray(
            Image.open(mask_path),
            dtype=np.int64
        )

        # Dataset may use 0/2 rather than 0/1. Convert to binary labels.
        mask = (mask > 0).astype(np.int64)

        image = torch.from_numpy(image).permute(2, 0, 1) / 255.0
        mask = torch.from_numpy(mask)

        return image, mask

seg_train_dataset = SegmentationDataset(train_seg_df)
seg_val_dataset = SegmentationDataset(val_seg_df)
seg_test_dataset = SegmentationDataset(test_seg_df)

seg_train_loader = DataLoader(seg_train_dataset, batch_size=4, shuffle=True)
seg_val_loader = DataLoader(seg_val_dataset, batch_size=4, shuffle=False)
seg_test_loader = DataLoader(seg_test_dataset, batch_size=4, shuffle=False)

image_tensor, mask_tensor = seg_train_dataset[0]

print("Image tensor:", image_tensor.shape, image_tensor.dtype)
print("Mask tensor:", mask_tensor.shape, mask_tensor.dtype)
print("Mask labels:", torch.unique(mask_tensor))


In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(10, 18))

for i in range(min(5, len(seg_train_dataset))):
    image, mask = seg_train_dataset[i]

    axes[i, 0].imshow(image.permute(1, 2, 0))
    axes[i, 0].set_title("Image")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(mask, cmap="gray")
    axes[i, 1].set_title("Ground-truth mask")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# ============================================
# PROPER U-NET FOR OIL-SPILL SEGMENTATION
# ============================================

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=2):
        super().__init__()

        # Encoder
        self.enc1 = DoubleConv(in_channels, 32)
        self.enc2 = DoubleConv(32, 64)
        self.enc3 = DoubleConv(64, 128)

        self.pool = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = DoubleConv(128, 256)

        # Decoder
        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec3 = DoubleConv(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = DoubleConv(128, 64)

        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec1 = DoubleConv(64, 32)

        # Final pixel classifier
        self.out = nn.Conv2d(32, num_classes, 1)

    def forward(self, x):

        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))

        # Bottleneck
        b = self.bottleneck(self.pool(e3))

        # Decoder + skip connections
        d3 = self.up3(b)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return self.out(d1)


seg_model = UNet(
    in_channels=3,
    num_classes=2
).to(DEVICE)

print(seg_model)

In [ ]:
# Check U-Net output shape

images, masks = next(iter(seg_train_loader))

images = images.to(DEVICE)

with torch.no_grad():
    outputs = seg_model(images)

print("Input shape:", images.shape)
print("Output shape:", outputs.shape)
print("Mask shape:", masks.shape)

In [ ]:
# ============================================
# SEGMENTATION LOSS: CROSS ENTROPY + DICE
# ============================================

class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):

        probs = torch.softmax(logits, dim=1)

        # Probability of oil class
        oil_probs = probs[:, 1]

        # Ground-truth oil pixels
        oil_targets = (targets == 1).float()

        oil_probs = oil_probs.reshape(-1)
        oil_targets = oil_targets.reshape(-1)

        intersection = (oil_probs * oil_targets).sum()

        dice = (
            2.0 * intersection + self.smooth
        ) / (
            oil_probs.sum()
            + oil_targets.sum()
            + self.smooth
        )

        return 1.0 - dice


class CombinedSegmentationLoss(nn.Module):
    def __init__(self):
        super().__init__()

        self.ce = nn.CrossEntropyLoss()
        self.dice = DiceLoss()

    def forward(self, logits, targets):

        ce_loss = self.ce(logits, targets)
        dice_loss = self.dice(logits, targets)

        return ce_loss + dice_loss


seg_criterion = CombinedSegmentationLoss()

seg_optimizer = torch.optim.Adam(
    seg_model.parameters(),
    lr=0.001
)

print("Segmentation loss created successfully!")
print("Optimizer created successfully!")

In [ ]:
# ============================================
# TRAIN PROPER U-NET
# ============================================

SEG_EPOCHS = 10

best_val_iou = -1.0
best_seg_path = SEG_OUTPUT_DIR / "best_unet_segmentation.pth"

for epoch in range(SEG_EPOCHS):

    seg_model.train()
    running_loss = 0.0

    for images, masks in seg_train_loader:

        images = images.to(DEVICE)
        masks = masks.to(DEVICE)

        seg_optimizer.zero_grad()

        outputs = seg_model(images)

        loss = seg_criterion(outputs, masks)

        loss.backward()
        seg_optimizer.step()

        running_loss += loss.item()

    # Validation only — NOT test data
    segmentation_metrics =()
    val_iou, val_dice = segmentation_metrics(
        seg_model,
        seg_val_loader
    )

    avg_train_loss = running_loss / len(seg_train_loader)

    print(
        f"Epoch {epoch + 1}/{SEG_EPOCHS} | "
        f"Train loss: {avg_train_loss:.4f} | "
        f"Val IoU: {val_iou:.4f} | "
        f"Val Dice: {val_dice:.4f}"
    )

    # Save best model using validation IoU
    if val_iou > best_val_iou:

        best_val_iou = val_iou

        torch.save(
            seg_model.state_dict(),
            best_seg_path
        )

        print("  → Best U-Net saved")


print()
print("Best validation IoU:", best_val_iou)
print("Best U-Net checkpoint:", best_seg_path)

In [ ]:
# ============================================
# FINAL U-NET TEST EVALUATION
# ============================================

# Load the BEST model selected using validation IoU
seg_model.load_state_dict(
    torch.load(
        best_seg_path,
        map_location=DEVICE
    )
)

seg_model.eval()

test_iou, test_dice = segmentation_metrics(
    seg_model,
    seg_test_loader
)

print("FINAL U-NET TEST RESULTS")
print("=" * 40)
print(f"Test IoU:  {test_iou:.4f}")
print(f"Test Dice: {test_dice:.4f}")

In [ ]:
# ============================================
# VISUALIZE U-NET TEST PREDICTIONS
# ============================================

import matplotlib.pyplot as plt

# Load best validation-selected model
seg_model.load_state_dict(
    torch.load(
        best_seg_path,
        map_location=DEVICE
    )
)

seg_model.eval()

# Get one test batch
images, masks = next(iter(seg_test_loader))

images_device = images.to(DEVICE)

with torch.no_grad():
    outputs = seg_model(images_device)
    predictions = torch.argmax(outputs, dim=1).cpu()

# Show 3 examples
fig, axes = plt.subplots(3, 3, figsize=(12, 12))

for i in range(3):

    image = images[i]

    # Convert CHW -> HWC
    image_display = image.permute(1, 2, 0).numpy()

    # Normalize for display
    image_display = (
        image_display - image_display.min()
    ) / (
        image_display.max() - image_display.min() + 1e-8
    )

    # Original image
    axes[i, 0].imshow(image_display)
    axes[i, 0].set_title("Original Image")
    axes[i, 0].axis("off")

    # Ground truth
    axes[i, 1].imshow(masks[i], cmap="gray")
    axes[i, 1].set_title("Ground Truth")
    axes[i, 1].axis("off")

    # Prediction
    axes[i, 2].imshow(predictions[i], cmap="gray")
    axes[i, 2].set_title("U-Net Prediction")
    axes[i, 2].axis("off")

plt.tight_layout()
plt.show()

### Segmentation model selection

The validation set is used to select the best checkpoint. The test set is
reserved for the final evaluation.


In [ ]:
def segmentation_metrics(model, loader):
    model.eval()
    intersection = union = pred_oil = true_oil = 0

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(DEVICE)
            masks = masks.to(DEVICE)

            predictions = model(images).argmax(dim=1)

            p = predictions == 1
            t = masks == 1

            intersection += (p & t).sum().item()
            union += (p | t).sum().item()
            pred_oil += p.sum().item()
            true_oil += t.sum().item()

    iou = intersection / (union + 1e-8)
    dice = 2 * intersection / (pred_oil + true_oil + 1e-8)

    return iou, dice


In [ ]:
SEG_EPOCHS = 5
best_val_iou = -1.0
best_seg_path = SEG_OUTPUT_DIR / "best_simple_segmentation_cnn.pth"

for epoch in range(SEG_EPOCHS):
    seg_model.train()
    running_loss = 0.0

    for images, masks in seg_train_loader:
        images = images.to(DEVICE)
        masks = masks.to(DEVICE)

        seg_optimizer.zero_grad()
        outputs = seg_model(images)
        loss = seg_criterion(outputs, masks)
        loss.backward()
        seg_optimizer.step()

        running_loss += loss.item()

    val_iou, val_dice = segmentation_metrics(seg_model, seg_val_loader)

    print(
        f"Epoch {epoch+1}/{SEG_EPOCHS} | "
        f"Train loss: {running_loss / len(seg_train_loader):.4f} | "
        f"Val IoU: {val_iou:.4f} | "
        f"Val Dice: {val_dice:.4f}"
    )

    if val_iou > best_val_iou:
        best_val_iou = val_iou
        torch.save(seg_model.state_dict(), best_seg_path)

print("Best segmentation checkpoint:", best_seg_path)


In [ ]:
seg_model.load_state_dict(
    torch.load(best_seg_path, map_location=DEVICE)
)

test_iou, test_dice = segmentation_metrics(
    seg_model,
    seg_test_loader
)

print("FINAL SEGMENTATION TEST")
print("IoU:", test_iou)
print("Dice:", test_dice)


## 3. CSIRO image-level Oil/No-Oil classification

This is a separate image-level task using the CSIRO dataset. Its class labels and model are not shared with pixel-level segmentation. The validation split is used for model selection; the test split is reserved for final evaluation.

# Experiment 2 — Classification

This is the CSIRO 5,630-chip dataset.

- Class 0 = no oil feature
- Class 1 = oil feature
- 3,725 class-0 images
- 1,905 class-1 images

The task is image-level classification. It does not create an oil boundary.


In [ ]:
CLS_DATA_DIR = CLS_DATASET_DIR / "data"

CLASS0_ZIP = CLS_DATA_DIR / "S1SAR_UnBalanced_400by400_Class_0.zip"
CLASS1_ZIP = CLS_DATA_DIR / "S1SAR_UnBalanced_400by400_Class_1.zip"

assert CLASS0_ZIP.exists(), f"Missing: {CLASS0_ZIP}"
assert CLASS1_ZIP.exists(), f"Missing: {CLASS1_ZIP}"

with zipfile.ZipFile(CLASS0_ZIP) as z:
    class0_files = sorted(
        f for f in z.namelist()
        if f.lower().endswith(".jpg")
    )

with zipfile.ZipFile(CLASS1_ZIP) as z:
    class1_files = sorted(
        f for f in z.namelist()
        if f.lower().endswith(".jpg")
    )

print("Class 0:", len(class0_files))
print("Class 1:", len(class1_files))
print("Total:", len(class0_files) + len(class1_files))


In [ ]:
with zipfile.ZipFile(CLASS0_ZIP) as z:
    image0 = Image.open(BytesIO(z.read(class0_files[0]))).convert("L")

with zipfile.ZipFile(CLASS1_ZIP) as z:
    image1 = Image.open(BytesIO(z.read(class1_files[0]))).convert("L")

print("Class 0:", image0.size, image0.mode)
print("Class 1:", image1.size, image1.mode)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(image0, cmap="gray")
axes[0].set_title("Class 0 — No Oil")
axes[0].axis("off")

axes[1].imshow(image1, cmap="gray")
axes[1].set_title("Class 1 — Oil")
axes[1].axis("off")

plt.tight_layout()
plt.show()


## Classification split

The old experiment used the test set while deciding which model/weight was
best. That is data leakage at the evaluation stage.

This notebook uses:

**Train → Validation → model selection → untouched Test**

The test set is evaluated only after the best validation checkpoint is chosen.


In [ ]:
all_cls_files = class0_files + class1_files
all_cls_labels = [0] * len(class0_files) + [1] * len(class1_files)

train_files, temp_files, train_labels, temp_labels = train_test_split(
    all_cls_files,
    all_cls_labels,
    test_size=0.30,
    random_state=SEED,
    stratify=all_cls_labels
)

val_files, test_files, val_labels, test_labels = train_test_split(
    temp_files,
    temp_labels,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_labels
)

print("Train:", len(train_files))
print("Validation:", len(val_files))
print("Test:", len(test_files))

print("Class counts:")
print("Train:", np.bincount(train_labels))
print("Validation:", np.bincount(val_labels))
print("Test:", np.bincount(test_labels))



## Related-crop leakage check

The filenames contain crop/image identifiers and geographic abbreviations.
A simple filename-based grouping check is performed below.

This is a conservative safeguard. A definitive scene-level leakage check
requires original-scene identifiers/provenance from the dataset.


In [ ]:
def crop_group_key(filename):
    name = Path(filename).name
    return re.sub(r"_cls_[01]\.jpg$", "", name)

def group_overlap(files_a, files_b):
    return (
        set(map(crop_group_key, files_a))
        & set(map(crop_group_key, files_b))
    )

print("Train ∩ Validation:", len(group_overlap(train_files, val_files)))
print("Train ∩ Test:", len(group_overlap(train_files, test_files)))
print("Validation ∩ Test:", len(group_overlap(val_files, test_files)))


In [ ]:
class CSIROOilDataset(Dataset):
    def __init__(self, files, labels):
        self.files = list(files)
        self.labels = list(labels)
        self.transform = T.ToTensor()

    def __len__(self):
        return len(self.files)

    def __getitem__(self, index):
        filename = self.files[index]
        label = int(self.labels[index])

        zip_path = CLASS0_ZIP if label == 0 else CLASS1_ZIP

        with zipfile.ZipFile(zip_path) as z:
            image = Image.open(
                BytesIO(z.read(filename))
            ).convert("L")

        return (
            self.transform(image),
            torch.tensor(label, dtype=torch.long)
        )

cls_train_dataset = CSIROOilDataset(train_files, train_labels)
cls_val_dataset = CSIROOilDataset(val_files, val_labels)
cls_test_dataset = CSIROOilDataset(test_files, test_labels)

cls_train_loader = DataLoader(
    cls_train_dataset, batch_size=32, shuffle=True
)
cls_val_loader = DataLoader(
    cls_val_dataset, batch_size=32, shuffle=False
)
cls_test_loader = DataLoader(
    cls_test_dataset, batch_size=32, shuffle=False
)

images, labels = next(iter(cls_train_loader))

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)
print("Image dtype:", images.dtype)
print("Label dtype:", labels.dtype)


In [ ]:
class StrongOilCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

cls_model = StrongOilCNN().to(DEVICE)

class_weights = torch.tensor(
    [1.0, 2.0],
    dtype=torch.float32,
    device=DEVICE
)

cls_criterion = nn.CrossEntropyLoss(weight=class_weights)
cls_optimizer = torch.optim.Adam(
    cls_model.parameters(),
    lr=1e-3
)

print(cls_model)
print("Class weights:", class_weights)


In [ ]:
def classification_metrics(model, loader):
    model.eval()

    y_true = []
    y_pred = []

    with torch.no_grad():
        for images, labels in loader:
            outputs = model(images.to(DEVICE))
            predictions = outputs.argmax(dim=1).cpu().numpy()

            y_pred.extend(predictions)
            y_true.extend(labels.numpy())

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(
            y_true, y_pred, zero_division=0
        ),
        "recall": recall_score(
            y_true, y_pred, zero_division=0
        ),
        "f1": f1_score(
            y_true, y_pred, zero_division=0
        )
    }


## Classification training

The checkpoint is selected using **validation F1**.

The test set is not used inside this training loop.


In [ ]:
CLS_EPOCHS = 10

best_val_f1 = -1.0
best_cls_path = CLS_OUTPUT_DIR / "best_strong_oil_cnn.pth"

for epoch in range(CLS_EPOCHS):
    cls_model.train()
    running_loss = 0.0

    for images, labels in cls_train_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        cls_optimizer.zero_grad()

        outputs = cls_model(images)
        loss = cls_criterion(outputs, labels)

        loss.backward()
        cls_optimizer.step()

        running_loss += loss.item()

    val_metrics = classification_metrics(
        cls_model,
        cls_val_loader
    )

    print(
        f"Epoch {epoch+1}/{CLS_EPOCHS} | "
        f"Train loss: {running_loss / len(cls_train_loader):.4f} | "
        f"Val F1: {val_metrics['f1']:.4f} | "
        f"Val recall: {val_metrics['recall']:.4f}"
    )

    if val_metrics["f1"] > best_val_f1:
        best_val_f1 = val_metrics["f1"]

        torch.save(
            cls_model.state_dict(),
            best_cls_path
        )

print("Best validation F1:", best_val_f1)
print("Saved checkpoint:", best_cls_path)


## Final classification test evaluation

Only now is the untouched test set used.


In [ ]:
cls_model.load_state_dict(
    torch.load(best_cls_path, map_location=DEVICE)
)

final_metrics = classification_metrics(
    cls_model,
    cls_test_loader
)

print("FINAL CLASSIFICATION TEST RESULTS")
for name, value in final_metrics.items():
    print(f"{name}: {value:.4f}")

y_true = []
y_pred = []

cls_model.eval()

with torch.no_grad():
    for images, labels in cls_test_loader:
        predictions = (
            cls_model(images.to(DEVICE))
            .argmax(dim=1)
            .cpu()
            .numpy()
        )

        y_pred.extend(predictions)
        y_true.extend(labels.numpy())

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))


## Saved-model inference

This function loads the saved classification checkpoint and predicts a
single grayscale SAR image as `Oil` or `No Oil`.

It is classification inference only and does not generate a spill outline.


In [ ]:
def load_classification_model(checkpoint_path=best_cls_path):
    model = StrongOilCNN().to(DEVICE)
    model.load_state_dict(
        torch.load(checkpoint_path, map_location=DEVICE)
    )
    model.eval()
    return model

def predict_classification(
    image_path,
    checkpoint_path=best_cls_path
):
    model = load_classification_model(checkpoint_path)

    image = Image.open(image_path).convert("L")
    tensor = T.ToTensor(image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        probabilities = torch.softmax(
            model(tensor),
            dim=1
        )[0]

    predicted_class = int(probabilities.argmax().item())

    return {
        "prediction": "Oil" if predicted_class == 1 else "No Oil",
        "probability_no_oil": float(probabilities[0].cpu()),
        "probability_oil": float(probabilities[1].cpu())
    }

print("Inference functions ready.")


# Final project interpretation

### What the two models do

**SimpleSegmentationCNN**
- Works on image + mask pairs.
- Produces a pixel-level prediction.
- Can be visualized as an oil-spill mask.
- This is the relevant direction for spill extent/outline and eventual drift work.

**StrongOilCNN**
- Works on complete 400×400 SAR chips.
- Produces `Oil` / `No Oil`.
- Does not locate the oil within the image.
- Does not produce a spill boundary.
- Therefore it should not be described as a drift/segmentation model.

### Evaluation correction

The earlier experiments compared models using test-set results. This clean
notebook instead selects the best classification checkpoint using validation
F1 and uses the test set only for final evaluation.

### Leakage caveat

The notebook performs a filename-based related-crop check. A stronger
scene-level split requires original-scene identifiers from the dataset
provenance if those identifiers are available.


## 4. Saved segmentation-model inference example

The following cells load a saved segmentation model and demonstrate prediction on one image. This is an example workflow, not a claim of a generally deployable inference pipeline.

In [ ]:
# ============================================
# LOAD SAVED U-NET AND PREDICT ON ONE IMAGE
# ============================================

from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

def load_unet_model(checkpoint_path):
    model = UNet(
        in_channels=3,
        num_classes=2
    ).to(DEVICE)

    model.load_state_dict(
        torch.load(
            checkpoint_path,
            map_location=DEVICE
        )
    )

    model.eval()
    return model


def predict_oil_mask(model, image_path):
    image = Image.open(image_path).convert("RGB")

    # Same size used by the segmentation dataset
    image_resized = image.resize((800, 600))

    image_array = np.array(image_resized).astype(np.float32) / 255.0

    image_tensor = torch.tensor(
        image_array
    ).permute(2, 0, 1).unsqueeze(0)

    image_tensor = image_tensor.to(DEVICE)

    with torch.no_grad():
        output = model(image_tensor)
        prediction = torch.argmax(
            output,
            dim=1
        )[0].cpu().numpy()

    return image_resized, prediction


# Load best saved model
loaded_unet = load_unet_model(best_seg_path)

print("Saved U-Net loaded successfully!")

In [ ]:
# ============================================
# EXAMPLE PREDICTION
# ============================================

sample_image_path = (
    SEG_IMAGE_DIR /
    "IMG_01_TILE_001.png"
)

print("Using image:", sample_image_path)
print("Exists:", sample_image_path.exists())

original_image, predicted_mask = predict_oil_mask(
    loaded_unet,
    sample_image_path
)

fig, axes = plt.subplots(
    1, 2,
    figsize=(12, 5)
)

axes[0].imshow(original_image)
axes[0].set_title("Input Image")
axes[0].axis("off")

axes[1].imshow(predicted_mask, cmap="gray")
axes[1].set_title("Predicted Oil Mask")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## 5. Sentinel-1 SAFE product exploration

Explore the SAFE archive structure, geolocation annotations, measurement rasters, polarisation bands and metadata. The SAFE ZIP is discovered in the notebook working directory; place it there or adapt the discovery cell.

In [ ]:
from pathlib import Path
import zipfile
import xml.etree.ElementTree as ET

# Find the Sentinel-1 ZIP in the project folder
zip_files = list(Path.cwd().glob("*.SAFE.zip"))
assert zip_files, "No .SAFE.zip file found in the notebook's working directory."

sentinel_zip = zip_files[0]
print("Product:", sentinel_zip.name)

def get_geolocation_points(archive, polarization):
    with zipfile.ZipFile(archive) as z:
        xml_files = [
            name for name in z.namelist()
            if "/annotation/" in name
            and "/calibration/" not in name
            and "/noise/" not in name
            and f"grd-{polarization}-" in name
            and name.endswith(".xml")
        ]

        assert len(xml_files) == 1, (
            f"Expected one {polarization.upper()} annotation; "
            f"found {len(xml_files)}"
        )

        root = ET.fromstring(z.read(xml_files[0]))

        return [
            tuple(
                next(child.text for child in point
                     if child.tag.split("}")[-1] == field)
                for field in ("line", "pixel", "latitude", "longitude")
            )
            for point in root.iter()
            if point.tag.split("}")[-1] == "geolocationGridPoint"
        ]

vv_points = get_geolocation_points(sentinel_zip, "vv")
vh_points = get_geolocation_points(sentinel_zip, "vh")

print("VV geolocation points:", len(vv_points))
print("VH geolocation points:", len(vh_points))
print("Identical geolocation grids:", vv_points == vh_points)

In [ ]:
import rasterio
import numpy as np

zip_path = str(sentinel_zip.resolve())
safe_folder = sentinel_zip.name.removesuffix(".zip")

def read_band_window(polarization, filename_suffix, window_size=512):
    tiff_path = (
        f"/vsizip/{zip_path}/{safe_folder}/measurement/"
        f"s1a-iw-grd-{polarization}-20211002t015821-"
        f"20211002t015850-039934-04b9c9-{filename_suffix}.tiff"
    )

    with rasterio.open(tiff_path) as src:
        return src.read(
            1,
            window=((0, window_size), (0, window_size))
        )

vv_patch = read_band_window("vv", "001")
vh_patch = read_band_window("vh", "002")

def describe_band(name, array):
    print(f"\n{name}")
    print("Shape:", array.shape)
    print("Minimum:", array.min())
    print("Maximum:", array.max())
    print("Mean:", round(float(array.mean()), 3))
    print("Median:", float(np.median(array)))
    print("Zero-pixel percentage:",
          round(float(np.mean(array == 0) * 100), 2), "%")

describe_band("VV patch", vv_patch)
describe_band("VH patch", vh_patch)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, image, title in zip(
    axes,
    [vv_patch, vh_patch],
    ["Sentinel-1 VV (raw DN)", "Sentinel-1 VH (raw DN)"]
):
    display_image = np.log1p(image.astype(np.float32))
    ax.imshow(display_image, cmap="gray")
    ax.set_title(title)
    ax.set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:
vv_large = read_band_window("vv", "001", window_size=1024)
vh_large = read_band_window("vh", "002", window_size=1024)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, image, title in zip(
    axes,
    [vv_large, vh_large],
    ["VV — 1024 × 1024", "VH — 1024 × 1024"]
):
    ax.imshow(
        np.log1p(image.astype(np.float32)),
        cmap="gray"
    )
    ax.set_title(title)
    ax.set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:
import zipfile

with zipfile.ZipFile(sentinel_zip) as z:
    names = z.namelist()

    for folder in [
        "/annotation/calibration/",
        "/annotation/noise/",
        "/annotation/"
    ]:
        matches = [
            n for n in names
            if folder in n and n.endswith(".xml")
        ]

        print(f"\n{folder}: {len(matches)} XML files")
        for name in matches[:4]:
            print(" ", name)

In [ ]:
import zipfile
import xml.etree.ElementTree as ET

def inspect_calibration_xml(polarization):
    with zipfile.ZipFile(sentinel_zip) as z:
        matches = [
            n for n in z.namelist()
            if "/annotation/calibration/" in n
            and f"calibration-s1a-iw-grd-{polarization}-" in n
            and n.endswith(".xml")
        ]

        assert len(matches) == 1, (
            f"Expected one {polarization.upper()} calibration file; "
            f"found {len(matches)}"
        )

        root = ET.fromstring(z.read(matches[0]))

        print(f"\n{polarization.upper()} calibration file")
        for element in root.iter():
            tag = element.tag.split("}")[-1]
            if tag in {
                "absoluteCalibrationConstant",
                "calibrationVector",
                "line",
                "pixel",
                "sigmaNought",
                "betaNought",
                "gamma",
            }:
                if tag != "calibrationVector":
                    print(f"{tag}: {element.text}")

        print("Calibration vectors:", sum(
            1 for element in root.iter()
            if element.tag.split("}")[-1] == "calibrationVector"
        ))

inspect_calibration_xml("vv")
inspect_calibration_xml("vh")

In [ ]:
import zipfile
import xml.etree.ElementTree as ET

def inspect_calibration_xml(polarization):
    with zipfile.ZipFile(sentinel_zip) as z:
        matches = [
            n for n in z.namelist()
            if "/annotation/calibration/" in n
            and f"calibration-s1a-iw-grd-{polarization}-" in n
            and n.endswith(".xml")
        ]

        assert len(matches) == 1, (
            f"Expected one {polarization.upper()} calibration file; "
            f"found {len(matches)}"
        )

        root = ET.fromstring(z.read(matches[0]))

        vectors = [
            element for element in root.iter()
            if element.tag.split("}")[-1] == "calibrationVector"
        ]

        print(f"\n{polarization.upper()} calibration file")
        print("Calibration vectors:", len(vectors))

inspect_calibration_xml("vv")
inspect_calibration_xml("vh")

In [ ]:
import zipfile
import xml.etree.ElementTree as ET

def inspect_one_calibration_vector(polarization):
    with zipfile.ZipFile(sentinel_zip) as z:
        matches = [
            n for n in z.namelist()
            if "/annotation/calibration/" in n
            and f"calibration-s1a-iw-grd-{polarization}-" in n
            and n.endswith(".xml")
        ]

        root = ET.fromstring(z.read(matches[0]))

        vector = next(
            element for element in root.iter()
            if element.tag.split("}")[-1] == "calibrationVector"
        )

        print(f"\n{polarization.upper()} — first calibration vector")

        for child in vector:
            tag = child.tag.split("}")[-1]
            values = (child.text or "").split()

            print(f"{tag}: {len(values)} values")
            print("  First 5:", values[:5])

inspect_one_calibration_vector("vv")
inspect_one_calibration_vector("vh")

In [ ]:
import zipfile
import xml.etree.ElementTree as ET

def summarize_calibration_vectors(polarization):
    with zipfile.ZipFile(sentinel_zip) as z:
        matches = [
            n for n in z.namelist()
            if "/annotation/calibration/" in n
            and f"calibration-s1a-iw-grd-{polarization}-" in n
            and n.endswith(".xml")
        ]

        root = ET.fromstring(z.read(matches[0]))

        vectors = [
            element for element in root.iter()
            if element.tag.split("}")[-1] == "calibrationVector"
        ]

        print(f"\n{polarization.upper()} calibration vectors")

        for vector in vectors:
            fields = {
                child.tag.split("}")[-1]: (child.text or "").split()
                for child in vector
            }

            line = fields.get("line", ["missing"])[0]
            pixels = fields.get("pixel", [])

            print(f"Line: {line:>6} | Pixel samples: {len(pixels)}")

summarize_calibration_vectors("vv")
summarize_calibration_vectors("vh")

In [ ]:
import zipfile
from pathlib import PurePosixPath

with zipfile.ZipFile(sentinel_zip) as z:
    measurement_files = [
        name for name in z.namelist()
        if "/measurement/" in name
        and PurePosixPath(name).suffix.lower() in {".tif", ".tiff"}
    ]

    for name in measurement_files:
        print(PurePosixPath(name).name)

In [ ]:
import zipfile
import rasterio
from rasterio.io import MemoryFile

with zipfile.ZipFile(sentinel_zip) as z:
    measurement_files = [
        name for name in z.namelist()
        if "/measurement/" in name
        and name.lower().endswith((".tif", ".tiff"))
    ]

    for name in measurement_files:
        print(f"\nFile: {name.split('/')[-1]}")

        with z.open(name) as source:
            with MemoryFile(source.read()) as memfile:
                with memfile.open() as dataset:
                    print("Width:", dataset.width)
                    print("Height:", dataset.height)
                    print("Bands:", dataset.count)
                    print("Data type:", dataset.dtypes)
                    print("CRS:", dataset.crs)

In [ ]:
import zipfile
import numpy as np
import rasterio
from rasterio.io import MemoryFile
from rasterio.windows import Window

with zipfile.ZipFile(sentinel_zip) as z:
    measurement_files = [
        name for name in z.namelist()
        if "/measurement/" in name
        and name.lower().endswith((".tif", ".tiff"))
    ]

    for name in measurement_files:
        print(f"\n{name.split('/')[-1]}")

        with z.open(name) as source:
            with MemoryFile(source.read()) as memfile:
                with memfile.open() as dataset:
                    window = Window(0, 0, 512, 512)
                    patch = dataset.read(1, window=window)

                    print("Patch shape:", patch.shape)
                    print("Minimum:", patch.min())
                    print("Maximum:", patch.max())
                    print("Mean:", round(float(patch.mean()), 2))
                    print("Standard deviation:",
                          round(float(patch.std()), 2))
                    print("Zero-valued pixels:",
                          int(np.count_nonzero(patch == 0)))

In [ ]:
import zipfile
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from rasterio.io import MemoryFile
from rasterio.windows import Window

patches = {}

with zipfile.ZipFile(sentinel_zip) as z:
    measurement_files = [
        name for name in z.namelist()
        if "/measurement/" in name
        and name.lower().endswith((".tif", ".tiff"))
    ]

    for name in measurement_files:
        polarization = "VH" if "-vh-" in name.lower() else "VV"

        with z.open(name) as source:
            with MemoryFile(source.read()) as memfile:
                with memfile.open() as dataset:
                    patches[polarization] = dataset.read(
                        1, window=Window(0, 0, 512, 512)
                    )

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, polarization in zip(axes, ["VH", "VV"]):
    ax.imshow(patches[polarization], cmap="gray")
    ax.set_title(f"{polarization} — raw pixel values")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

for polarization in ["VH", "VV"]:
    patch = patches[polarization]

    zero_per_row = np.count_nonzero(patch == 0, axis=1)

    print(f"\n{polarization}")
    print("Zero pixels in first row:", zero_per_row[0])
    print("Zero pixels in first 10 rows:", zero_per_row[:10].tolist())
    print("Rows with more than 50% zero pixels:",
          int(np.count_nonzero(zero_per_row > patch.shape[1] / 2)))

In [ ]:
for polarization in ["VH", "VV"]:
    patch = patches[polarization]

    zero_per_row = np.count_nonzero(patch == 0, axis=1)
    valid_rows = np.flatnonzero(zero_per_row < patch.shape[1])

    print(f"\n{polarization}")
    print("First row containing any non-zero pixels:",
          int(valid_rows[0]) if len(valid_rows) else "None")
    print("Zero pixels in that row:",
          int(zero_per_row[valid_rows[0]]) if len(valid_rows) else "N/A")

In [ ]:
for polarization in ["VH", "VV"]:
    patch = patches[polarization]

    zero_per_column = np.count_nonzero(patch == 0, axis=0)

    print(f"\n{polarization}")
    print("Zero pixels in first 10 columns:",
          zero_per_column[:10].tolist())
    print("Zero pixels in last 10 columns:",
          zero_per_column[-10:].tolist())
    print("Columns with more than 50% zero pixels:",
          int(np.count_nonzero(zero_per_column > patch.shape[0] / 2)))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, polarization in zip(axes, ["VH", "VV"]):
    zero_mask = patches[polarization] == 0

    ax.imshow(zero_mask, cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"{polarization} — zero-valued pixels")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, polarization in zip(axes, ["VH", "VV"]):
    zero_mask = patches[polarization] == 0

    ax.imshow(zero_mask, cmap="gray_r", vmin=0, vmax=1)
    ax.set_title(
        f"{polarization} — zero pixels: {zero_mask.sum():,}"
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, polarization in zip(axes, ["VH", "VV"]):
    patch = patches[polarization].astype(np.float32)

    # Exclude zero-valued pixels when calculating display limits
    valid = patch[patch > 0]
    low, high = np.percentile(valid, [2, 98])

    display_patch = np.ma.masked_equal(patch, 0)

    ax.imshow(
        display_patch,
        cmap="gray",
        vmin=low,
        vmax=high
    )
    ax.set_title(f"{polarization} — contrast enhanced")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import zipfile
import xml.etree.ElementTree as ET

with zipfile.ZipFile(sentinel_zip) as z:
    for polarization in ["vv", "vh"]:
        matches = [
            name for name in z.namelist()
            if "/annotation/calibration/" in name
            and f"calibration-s1a-iw-grd-{polarization}-" in name
            and name.endswith(".xml")
        ]

        root = ET.fromstring(z.read(matches[0]))

        print(f"\n{polarization.upper()}")

        constants = [
            element.text
            for element in root.iter()
            if element.tag.split("}")[-1] == "absoluteCalibrationConstant"
        ]

        print("Absolute calibration constant:", constants)

In [ ]:
import zipfile
import xml.etree.ElementTree as ET

with zipfile.ZipFile(sentinel_zip) as z:
    metadata_files = [
        name for name in z.namelist()
        if "/annotation/" in name
        and name.endswith(".xml")
        and "/calibration/" not in name
        and "/noise/" not in name
    ]

    print("Annotation XML files:")
    for name in metadata_files:
        print(name.split("/")[-1])

In [ ]:
import zipfile
import xml.etree.ElementTree as ET

with zipfile.ZipFile(sentinel_zip) as z:
    for polarization in ["vv", "vh"]:
        matches = [
            name for name in z.namelist()
            if "/annotation/" in name
            and "/calibration/" not in name
            and "/noise/" not in name
            and f"grd-{polarization}-" in name
            and name.endswith(".xml")
        ]

        root = ET.fromstring(z.read(matches[0]))

        print(f"\n{polarization.upper()} annotation metadata")

        wanted_tags = {
            "product",
            "numberOfLines",
            "numberOfSamples",
            "rangePixelSpacing",
            "azimuthPixelSpacing",
            "projection",
            "geolocationGridPoint",
        }

        for element in root.iter():
            tag = element.tag.split("}")[-1]

            if tag in wanted_tags and tag != "geolocationGridPoint":
                print(f"{tag}: {element.text}")

        print(
            "Geolocation grid points:",
            sum(
                1 for element in root.iter()
                if element.tag.split("}")[-1] == "geolocationGridPoint"
            )
        )

In [ ]:
import zipfile
import xml.etree.ElementTree as ET

with zipfile.ZipFile(sentinel_zip) as z:
    name = next(
        n for n in z.namelist()
        if "/annotation/" in n
        and "/calibration/" not in n
        and "/noise/" not in n
        and "grd-vv-" in n
        and n.endswith(".xml")
    )

    root = ET.fromstring(z.read(name))

    points = [
        element for element in root.iter()
        if element.tag.split("}")[-1] == "geolocationGridPoint"
    ]

    print("Total grid points:", len(points))
    print("\nFirst 5 geolocation points:")

    for point in points[:5]:
        values = {
            child.tag.split("}")[-1]: child.text
            for child in point
        }

        print({
            key: values.get(key)
            for key in [
                "line", "pixel", "latitude", "longitude",
                "incidenceAngle"
            ]
        })

In [ ]:
import zipfile
import xml.etree.ElementTree as ET

with zipfile.ZipFile(sentinel_zip) as z:
    name = next(
        n for n in z.namelist()
        if "/annotation/" in n
        and "/calibration/" not in n
        and "/noise/" not in n
        and "grd-vv-" in n
        and n.endswith(".xml")
    )

    root = ET.fromstring(z.read(name))

    points = [
        element for element in root.iter()
        if element.tag.split("}")[-1] == "geolocationGridPoint"
    ]

    latitudes = []
    longitudes = []

    for point in points:
        values = {
            child.tag.split("}")[-1]: child.text
            for child in point
        }

        if values.get("latitude") and values.get("longitude"):
            latitudes.append(float(values["latitude"]))
            longitudes.append(float(values["longitude"]))

    print("Grid points with coordinates:", len(latitudes))
    print("Minimum latitude:", min(latitudes))
    print("Maximum latitude:", max(latitudes))
    print("Minimum longitude:", min(longitudes))
    print("Maximum longitude:", max(longitudes))

In [ ]:
import zipfile
import xml.etree.ElementTree as ET

with zipfile.ZipFile(sentinel_zip) as z:
    calibration_files = [
        name for name in z.namelist()
        if "calibration" in name.lower()
        and name.lower().endswith(".xml")
    ]

print("Calibration XML files:")
for name in calibration_files:
    print(name)

In [ ]:
import zipfile
import xml.etree.ElementTree as ET

with zipfile.ZipFile(sentinel_zip) as z:
    for name in calibration_files:
        if "/calibration-s1a-" not in name.lower():
            continue

        root = ET.fromstring(z.read(name))

        def get_text(tag):
            element = root.find(f".//{{*}}{tag}")
            return element.text.strip() if element is not None and element.text else "Not found"

        print("\nFile:", name.split("/")[-1])
        print("Product:", get_text("product"))
        print("Absolute calibration constant:", get_text("absoluteCalibrationConstant"))

        vectors = root.findall(".//{*}calibrationVector")
        print("Number of calibration vectors:", len(vectors))

        if vectors:
            vector = vectors[0]
            for tag in ["line", "pixel", "sigmaNought", "betaNought", "gamma", "dn"]:
                element = vector.find(f"{{*}}{tag}")
                if element is not None and element.text:
                    values = element.text.split()
                    print(f"{tag}: {len(values)} values; first 5:", values[:5])

In [ ]:
import zipfile
import xml.etree.ElementTree as ET
import numpy as np

calibration_data = {}

with zipfile.ZipFile(sentinel_zip) as z:
    for name in calibration_files:
        filename = name.split("/")[-1].lower()

        if not filename.startswith("calibration-s1a-"):
            continue

        polarization = "-vh-" if "-vh-" in filename else "-vv-"
        pol = polarization.strip("-")

        root = ET.fromstring(z.read(name))
        vectors = root.findall(".//{*}calibrationVector")

        calibration_data[pol] = {}

        for field in ["sigmaNought", "betaNought", "gamma", "dn"]:
            all_values = []

            for vector in vectors:
                element = vector.find(f"{{*}}{field}")
                if element is not None and element.text:
                    all_values.extend(
                        map(float, element.text.split())
                    )

            calibration_data[pol][field] = np.array(all_values)

for field in ["sigmaNought", "betaNought", "gamma", "dn"]:
    vh = calibration_data["vh"][field]
    vv = calibration_data["vv"][field]

    print(f"\n{field}")
    print("VH values:", len(vh), "| VV values:", len(vv))
    print("Arrays identical:", np.array_equal(vh, vv))

    if len(vh) == len(vv):
        print("Maximum absolute difference:", np.max(np.abs(vh - vv)))

In [ ]:
import zipfile
import xml.etree.ElementTree as ET

with zipfile.ZipFile(sentinel_zip) as z:
    for name in z.namelist():
        filename = name.split("/")[-1].lower()

        if not filename.startswith("noise-s1a-") or not filename.endswith(".xml"):
            continue

        root = ET.fromstring(z.read(name))

        print("\nFile:", filename)
        print("Root element:", root.tag.split("}")[-1])

        # Show the XML element names and a few example values
        shown = 0
        for element in root.iter():
            if element is root:
                continue

            value = (element.text or "").strip()
            if value:
                print(f"{element.tag.split('}')[-1]}: {value[:120]}")
                shown += 1

            if shown >= 12:
                break

In [ ]:
import zipfile
import xml.etree.ElementTree as ET

with zipfile.ZipFile(sentinel_zip) as z:
    for name in z.namelist():
        filename = name.split("/")[-1].lower()

        if not filename.startswith("noise-s1a-") or not filename.endswith(".xml"):
            continue

        root = ET.fromstring(z.read(name))

        print("\nFile:", filename)

        for tag in ["noiseRangeVector", "noiseAzimuthVector"]:
            vectors = root.findall(f".//{{*}}{tag}")
            print(f"{tag}: {len(vectors)} vectors")

        # Show the first noise range vector, if present
        vector = root.find(".//{*}noiseRangeVector")

        if vector is not None:
            for child in vector:
                value = (child.text or "").strip()
                if value:
                    print(
                        child.tag.split("}")[-1],
                        "sample:",
                        value[:160]
                    )

In [ ]:
import zipfile
import rasterio
from rasterio.io import MemoryFile

with zipfile.ZipFile(sentinel_zip) as z:
    measurement_files = [
        name for name in z.namelist()
        if name.lower().endswith((".tif", ".tiff"))
        and ("-vh-" in name.lower() or "-vv-" in name.lower())
    ]

    for name in measurement_files:
        print("\nPolarization file:", name.split("/")[-1])

        with MemoryFile(z.read(name)) as memfile:
            with memfile.open() as src:
                positions = [
                    (0, 0),
                    (0, 5000),
                    (5000, 5000),
                    (10000, 10000),
                    (src.height - 512, src.width - 512),
                ]

                for row, col in positions:
                    window = rasterio.windows.Window(
                        col, row, 512, 512
                    )
                    patch = src.read(1, window=window)

                    print(
                        f"row={row}, col={col}: "
                        f"min={patch.min()}, max={patch.max()}, "
                        f"mean={patch.mean():.2f}, "
                        f"zero_pixels={(patch == 0).sum()}"
                    )

In [ ]:
import zipfile
import tempfile
import os
import numpy as np
import rasterio
import matplotlib.pyplot as plt

with zipfile.ZipFile(sentinel_zip) as z:
    measurement_files = {
        "VH": next(
            n for n in z.namelist()
            if n.lower().endswith(".tiff") and "-vh-" in n.lower()
        ),
        "VV": next(
            n for n in z.namelist()
            if n.lower().endswith(".tiff") and "-vv-" in n.lower()
        ),
    }

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))

    for row_idx, (pol, name) in enumerate(measurement_files.items()):
        with tempfile.TemporaryDirectory() as temp_dir:
            local_path = os.path.join(temp_dir, os.path.basename(name))

            with z.open(name) as source, open(local_path, "wb") as target:
                while True:
                    chunk = source.read(1024 * 1024)
                    if not chunk:
                        break
                    target.write(chunk)

            with rasterio.open(local_path) as src:
                patch = src.read(
                    1,
                    window=rasterio.windows.Window(
                        src.width - 512,
                        src.height - 512,
                        512,
                        512,
                    ),
                )

        valid = patch[patch > 0]
        vmin, vmax = np.percentile(valid, [2, 98])

        axes[row_idx, 0].imshow(
            patch, cmap="gray", vmin=vmin, vmax=vmax
        )
        axes[row_idx, 0].set_title(f"{pol}: bottom-right intensity")

        axes[row_idx, 1].imshow(patch == 0, cmap="gray")
        axes[row_idx, 1].set_title(f"{pol}: zero-valued pixels")

        for ax in axes[row_idx]:
            ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
import zipfile
import tempfile
import os
import numpy as np
import rasterio

with zipfile.ZipFile(sentinel_zip) as z:
    for pol in ["vh", "vv"]:
        name = next(
            n for n in z.namelist()
            if n.lower().endswith(".tiff")
            and f"-{pol}-" in n.lower()
        )

        with tempfile.TemporaryDirectory() as temp_dir:
            path = os.path.join(temp_dir, os.path.basename(name))

            with z.open(name) as source, open(path, "wb") as target:
                while True:
                    chunk = source.read(1024 * 1024)
                    if not chunk:
                        break
                    target.write(chunk)

            with rasterio.open(path) as src:
                patch = src.read(
                    1,
                    window=rasterio.windows.Window(
                        src.width - 512,
                        src.height - 512,
                        512,
                        512,
                    ),
                )

        # Compare the left and right halves of the patch
        left = patch[:, :256]
        right = patch[:, 256:]

        print(f"\n{pol.upper()}")
        print(
            "Left half:",
            "min =", left.min(),
            "max =", left.max(),
            "mean =", round(float(left.mean()), 2),
            "zeros =", int((left == 0).sum()),
        )
        print(
            "Right half:",
            "min =", right.min(),
            "max =", right.max(),
            "mean =", round(float(right.mean()), 2),
            "zeros =", int((right == 0).sum()),
        )

In [ ]:
import zipfile
import tempfile
import os
import numpy as np
import rasterio

with zipfile.ZipFile(sentinel_zip) as z:
    name = next(
        n for n in z.namelist()
        if n.lower().endswith(".tiff") and "-vh-" in n.lower()
    )

    with tempfile.TemporaryDirectory() as temp_dir:
        path = os.path.join(temp_dir, os.path.basename(name))

        with z.open(name) as source, open(path, "wb") as target:
            while True:
                chunk = source.read(1024 * 1024)
                if not chunk:
                    break
                target.write(chunk)

        with rasterio.open(path) as src:
            patch = src.read(
                1,
                window=rasterio.windows.Window(
                    src.width - 512,
                    src.height - 512,
                    512,
                    512,
                ),
            )

zero_counts = (patch == 0).sum(axis=1)

print("First 20 row zero counts:")
print(zero_counts[:20])

print("\nLast 20 row zero counts:")
print(zero_counts[-20:])

print("\nRows with all 512 pixels zero:", int((zero_counts == 512).sum()))
print("Rows with at least one zero:", int((zero_counts > 0).sum()))

In [ ]:
zero_counts_by_column = (patch == 0).sum(axis=0)

print("First 20 column zero counts:")
print(zero_counts_by_column[:20])

print("\nLast 20 column zero counts:")
print(zero_counts_by_column[-20:])

print("\nColumns with all 512 pixels zero:",
      int((zero_counts_by_column == 512).sum()))

print("Columns with at least one zero:",
      int((zero_counts_by_column > 0).sum()))

print("Columns with exactly 512 nonzero pixels:",
      int((zero_counts_by_column == 0).sum()))

In [ ]:
import zipfile
import tempfile
import os
import numpy as np
import rasterio
import matplotlib.pyplot as plt

with zipfile.ZipFile(sentinel_zip) as z:
    name = next(
        n for n in z.namelist()
        if n.lower().endswith(".tiff") and "-vh-" in n.lower()
    )

    with tempfile.TemporaryDirectory() as temp_dir:
        path = os.path.join(temp_dir, os.path.basename(name))

        with z.open(name) as source, open(path, "wb") as target:
            while True:
                chunk = source.read(1024 * 1024)
                if not chunk:
                    break
                target.write(chunk)

        with rasterio.open(path) as src:
            overview = src.read(
                1,
                out_shape=(600, 800),
                resampling=rasterio.enums.Resampling.nearest,
            )

zero_mask = overview == 0

plt.figure(figsize=(12, 7))
plt.imshow(zero_mask, cmap="gray", interpolation="nearest")
plt.title("VH full-scene overview: zero-valued pixels")
plt.xlabel("Overview column")
plt.ylabel("Overview row")
plt.axis("off")
plt.show()

print("Overview shape:", overview.shape)
print("Zero-valued fraction:", round(float(zero_mask.mean()) * 100, 2), "%")

In [ ]:
plt.figure(figsize=(12, 7))
plt.imshow(zero_mask, cmap="gray", vmin=0, vmax=1)
plt.title("White = zero-valued pixels | Black = nonzero pixels")
plt.axis("off")
plt.show()

print("Zero pixels:", int(zero_mask.sum()))
print("Nonzero pixels:", int((~zero_mask).sum()))
print("Total pixels:", zero_mask.size)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

valid = overview[overview > 0]
vmin, vmax = np.percentile(valid, [2, 98])

axes[0].imshow(overview, cmap="gray", vmin=vmin, vmax=vmax)
axes[0].set_title("VH intensity overview")

axes[1].imshow(zero_mask, cmap="gray", vmin=0, vmax=1)
axes[1].set_title("Zero mask: white = zero, black = nonzero")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import zipfile
import tempfile
import os
import numpy as np
import rasterio
from rasterio.windows import Window

with zipfile.ZipFile(sentinel_zip) as z:
    name = next(
        n for n in z.namelist()
        if n.lower().endswith(".tiff") and "-vh-" in n.lower()
    )

    with tempfile.TemporaryDirectory() as temp_dir:
        path = os.path.join(temp_dir, os.path.basename(name))

        with z.open(name) as source, open(path, "wb") as target:
            while True:
                chunk = source.read(1024 * 1024)
                if not chunk:
                    break
                target.write(chunk)

        with rasterio.open(path) as src:
            total_pixels = src.width * src.height
            zero_pixels = 0

            window_size = 1024

            for row in range(0, src.height, window_size):
                for col in range(0, src.width, window_size):
                    window = Window(
                        col,
                        row,
                        min(window_size, src.width - col),
                        min(window_size, src.height - row),
                    )

                    patch = src.read(1, window=window)
                    zero_pixels += int((patch == 0).sum())

            print("Image dimensions:", src.width, "x", src.height)
            print("Total pixels:", total_pixels)
            print("Zero-valued pixels:", zero_pixels)
            print("Nonzero pixels:", total_pixels - zero_pixels)
            print("Zero-valued percentage:",
                  round(100 * zero_pixels / total_pixels, 4), "%")

In [ ]:
import zipfile
import tempfile
import os
import numpy as np
import rasterio
from rasterio.windows import Window

with zipfile.ZipFile(sentinel_zip) as z:
    files = {
        "VH": next(
            n for n in z.namelist()
            if n.lower().endswith(".tiff") and "-vh-" in n.lower()
        ),
        "VV": next(
            n for n in z.namelist()
            if n.lower().endswith(".tiff") and "-vv-" in n.lower()
        ),
    }

    with tempfile.TemporaryDirectory() as temp_dir:
        paths = {}

        for pol, name in files.items():
            path = os.path.join(temp_dir, os.path.basename(name))
            with z.open(name) as source, open(path, "wb") as target:
                while True:
                    chunk = source.read(1024 * 1024)
                    if not chunk:
                        break
                    target.write(chunk)
            paths[pol] = path

        with rasterio.open(paths["VH"]) as vh_src, \
             rasterio.open(paths["VV"]) as vv_src:

            print("Same dimensions:", 
                  vh_src.width == vv_src.width and
                  vh_src.height == vv_src.height)

            vh_zero = 0
            vv_zero = 0
            both_zero = 0
            different_zero_status = 0

            size = 1024

            for row in range(0, vh_src.height, size):
                for col in range(0, vh_src.width, size):
                    window = Window(
                        col, row,
                        min(size, vh_src.width - col),
                        min(size, vh_src.height - row),
                    )

                    vh = vh_src.read(1, window=window)
                    vv = vv_src.read(1, window=window)

                    vh_mask = vh == 0
                    vv_mask = vv == 0

                    vh_zero += int(vh_mask.sum())
                    vv_zero += int(vv_mask.sum())
                    both_zero += int((vh_mask & vv_mask).sum())
                    different_zero_status += int(
                        (vh_mask != vv_mask).sum()
                    )

            print("VH zero pixels:", vh_zero)
            print("VV zero pixels:", vv_zero)
            print("Zero in both channels:", both_zero)
            print("Pixels with different zero status:",
                  different_zero_status)

In [ ]:
import zipfile
import tempfile
import os
import numpy as np
import rasterio
from rasterio.windows import Window

with zipfile.ZipFile(sentinel_zip) as z:
    files = {
        "VH": next(
            n for n in z.namelist()
            if n.lower().endswith(".tiff") and "-vh-" in n.lower()
        ),
        "VV": next(
            n for n in z.namelist()
            if n.lower().endswith(".tiff") and "-vv-" in n.lower()
        ),
    }

    with tempfile.TemporaryDirectory() as temp_dir:
        paths = {}

        for pol, name in files.items():
            path = os.path.join(temp_dir, os.path.basename(name))
            with z.open(name) as source, open(path, "wb") as target:
                while True:
                    chunk = source.read(1024 * 1024)
                    if not chunk:
                        break
                    target.write(chunk)
            paths[pol] = path

        with rasterio.open(paths["VH"]) as vh_src, \
             rasterio.open(paths["VV"]) as vv_src:

            print("Same dimensions:", 
                  vh_src.width == vv_src.width and
                  vh_src.height == vv_src.height)

            vh_zero = 0
            vv_zero = 0
            both_zero = 0
            different_zero_status = 0

            size = 1024

            for row in range(0, vh_src.height, size):
                for col in range(0, vh_src.width, size):
                    window = Window(
                        col, row,
                        min(size, vh_src.width - col),
                        min(size, vh_src.height - row),
                    )

                    vh = vh_src.read(1, window=window)
                    vv = vv_src.read(1, window=window)

                    vh_mask = vh == 0
                    vv_mask = vv == 0

                    vh_zero += int(vh_mask.sum())
                    vv_zero += int(vv_mask.sum())
                    both_zero += int((vh_mask & vv_mask).sum())
                    different_zero_status += int(
                        (vh_mask != vv_mask).sum()
                    )

            print("VH zero pixels:", vh_zero)
            print("VV zero pixels:", vv_zero)
            print("Zero in both channels:", both_zero)
            print("Pixels with different zero status:",
                  different_zero_status)

In [ ]:
import zipfile
import tempfile
import os
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from rasterio.windows import Window

with zipfile.ZipFile(sentinel_zip) as z:
    names = {
        "VH": next(
            n for n in z.namelist()
            if n.lower().endswith(".tiff") and "-vh-" in n.lower()
        ),
        "VV": next(
            n for n in z.namelist()
            if n.lower().endswith(".tiff") and "-vv-" in n.lower()
        ),
    }

    with tempfile.TemporaryDirectory() as temp_dir:
        paths = {}

        for pol, name in names.items():
            path = os.path.join(temp_dir, os.path.basename(name))

            with z.open(name) as source, open(path, "wb") as target:
                while True:
                    chunk = source.read(1024 * 1024)
                    if not chunk:
                        break
                    target.write(chunk)

            paths[pol] = path

        with rasterio.open(paths["VH"]) as vh_src, \
             rasterio.open(paths["VV"]) as vv_src:

            window = Window(5000, 5000, 512, 512)
            vh = vh_src.read(1, window=window)
            vv = vv_src.read(1, window=window)

valid_mask = vh != 0

print("Patch shape:", valid_mask.shape)
print("Valid pixels:", int(valid_mask.sum()))
print("Zero-valued pixels:", int((~valid_mask).sum()))
print("VH and VV zero masks match:",
      bool(np.array_equal(vh == 0, vv == 0)))

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(vh, cmap="gray")
axes[0].set_title("VH patch")
axes[1].imshow(vv, cmap="gray")
axes[1].set_title("VV patch")
axes[2].imshow(valid_mask, cmap="gray", vmin=0, vmax=1)
axes[2].set_title("Shared mask: white = valid")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import zipfile

with zipfile.ZipFile(sentinel_zip) as z:
    candidates = [
        name for name in z.namelist()
        if any(
            word in name.lower()
            for word in [
                "annotation",
                "manifest",
                "mask",
                "label",
                "ground_truth",
                "ground-truth",
                "oil",
                "slick",
            ]
        )
    ]

print("Potential annotation or label files:", len(candidates))

for name in candidates[:100]:
    print(name)

if len(candidates) > 100:
    print("...and", len(candidates) - 100, "more")

## 6. Segmentation dataset inventory and pipeline inspection

Inspect the segmentation project layout, split CSVs, image/mask values, label coverage and pipeline scripts/configuration. These cells inspect existing files and do not change dataset contents.

In [ ]:
from pathlib import Path

project_dir = Path.cwd()

print("Current project folder:", project_dir)
print("\nFiles and folders:")

for item in sorted(project_dir.iterdir()):
    if item.is_dir():
        print("[DIR] ", item.name)
    else:
        print("[FILE]", item.name)

In [ ]:
from pathlib import Path

dataset_dir = Path.cwd() / "oilspill-detection-dataset_20260912-1b9a62c"

print("Dataset folder:", dataset_dir)

for item in sorted(dataset_dir.rglob("*")):
    if item.is_file():
        print(item.relative_to(dataset_dir))

In [ ]:
from pathlib import Path

dataset_dir = Path.cwd() / "oilspill-detection-dataset_20260912-1b9a62c"

for folder in sorted(p for p in dataset_dir.rglob("*") if p.is_dir()):
    files = [p for p in folder.iterdir() if p.is_file()]
    print(f"{folder.relative_to(dataset_dir)}: {len(files)} files")

print("\nTop-level split files:")
for name in ["train.csv", "val.csv", "test.csv"]:
    path = dataset_dir / "splits" / name
    print(f"{name}: {'exists' if path.exists() else 'missing'}")

In [ ]:
from pathlib import Path

dataset_dir = Path.cwd() / "oilspill-detection-dataset_20260912-1b9a62c"
card_path = dataset_dir / "DATASET_CARD.md"

print(card_path.read_text(encoding="utf-8"))

In [ ]:
from pathlib import Path
import pandas as pd

dataset_dir = Path.cwd() / "oilspill-detection-dataset_20260912-1b9a62c"

train_df = pd.read_csv(dataset_dir / "splits" / "train.csv")
val_df = pd.read_csv(dataset_dir / "splits" / "val.csv")
test_df = pd.read_csv(dataset_dir / "splits" / "test.csv")

print("Train samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Test samples:", len(test_df))

print("\nTrain manifest columns:")
print(train_df.columns.tolist())

print("\nFirst 3 training samples:")
display(train_df.head(3))

print("\nFirst sample image exists:",
      (dataset_dir / train_df.iloc[0]["image_path"]).exists())

print("First sample mask exists:",
      (dataset_dir / train_df.iloc[0]["mask_path"]).exists())

In [ ]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

# Get the first training image and its corresponding mask
image_path = dataset_dir / train_df.iloc[0]["image_path"]
mask_path = dataset_dir / train_df.iloc[0]["mask_path"]

image = np.array(Image.open(image_path).convert("RGB"))
mask = np.array(Image.open(mask_path))

print("Image shape:", image.shape)
print("Image data type:", image.dtype)

print("\nMask shape:", mask.shape)
print("Mask data type:", mask.dtype)
print("Unique mask values:", np.unique(mask, return_counts=True))

# Display the image and mask side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(image)
axes[0].set_title("Input SAR Image")
axes[0].axis("off")

axes[1].imshow(mask, cmap="gray", vmin=0, vmax=1)
axes[1].set_title("Ground-Truth Mask (0 = Sea, 1 = Slick)")
axes[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
print("Image shape:", image.shape)
print("Mask shape:", mask.shape)

print("Unique mask values:", np.unique(mask))
print("Mask pixel counts:", np.unique(mask, return_counts=True))

print("Mask data type:", mask.dtype)

In [ ]:
from collections import Counter
import numpy as np
from PIL import Image

mask_value_counts = Counter()

for _, row in pd.concat(
    [train_df, val_df, test_df], ignore_index=True
).iterrows():
    mask_path = dataset_dir / row["mask_path"]

    with Image.open(mask_path) as mask_image:
        mask_array = np.array(mask_image)

    mask_value_counts.update(
        np.unique(mask_array).tolist()
    )

print("Mask values found across the dataset:")
print(dict(mask_value_counts))

In [ ]:
from collections import Counter
import numpy as np
from PIL import Image

mask_patterns = Counter()
examples = {}

all_splits = pd.concat(
    [train_df, val_df, test_df],
    ignore_index=True
)

for _, row in all_splits.iterrows():
    mask_path = dataset_dir / row["mask_path"]

    with Image.open(mask_path) as mask_image:
        values = tuple(np.unique(np.array(mask_image)).tolist())

    mask_patterns[values] += 1
    examples.setdefault(values, row["sample_id"])

print("Mask value patterns and number of masks:")
for pattern, count in sorted(mask_patterns.items()):
    print(f"{pattern}: {count} masks; example: {examples[pattern]}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

sample_id = "IMG_02_TILE_006"

row = all_splits.loc[
    all_splits["sample_id"] == sample_id
].iloc[0]

image = np.array(
    Image.open(dataset_dir / row["image_path"]).convert("RGB")
)

mask = np.array(
    Image.open(dataset_dir / row["mask_path"])
)

print("Image shape:", image.shape)
print("Mask shape:", mask.shape)

for value in np.unique(mask):
    print(f"Label {value}: {np.sum(mask == value):,} pixels")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(image)
axes[0].set_title("Input SAR Image")
axes[0].axis("off")

axes[1].imshow(mask, cmap="viridis", vmin=0, vmax=2)
axes[1].set_title("Original mask: labels 0, 1, 2")
axes[1].axis("off")

axes[2].imshow(image)
axes[2].imshow(
    np.ma.masked_where(mask == 0, mask),
    cmap="autumn",
    alpha=0.75,
    vmin=1,
    vmax=2
)
axes[2].set_title("Label overlay")
axes[2].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path

search_dirs = [
    dataset_dir,
    dataset_dir / "pipeline_ml",
    dataset_dir / "pipeline_classical",
    dataset_dir / "pipeline_hybrid",
]

keywords = ("label", "mask", "slick", "seepage", "classe", "class")

for folder in search_dirs:
    if not folder.exists():
        continue

    for path in folder.rglob("*"):
        if not path.is_file() or path.suffix.lower() not in {
            ".py", ".md", ".json", ".yaml", ".yml", ".csv"
        }:
            continue

        try:
            lines = path.read_text(
                encoding="utf-8", errors="ignore"
            ).splitlines()
        except OSError:
            continue

        matches = [
            (i, line.strip())
            for i, line in enumerate(lines, start=1)
            if any(word in line.lower() for word in keywords)
            and any(token in line for token in ("= 1", "= 2", "== 1", "== 2", "0, 1, 2"))
        ]

        if matches:
            print(f"\n--- {path.relative_to(dataset_dir)} ---")
            for line_no, line in matches[:8]:
                print(f"{line_no}: {line[:220]}")

In [ ]:
common_path = dataset_dir / "pipeline_ml" / "scripts" / "common.py"

print(common_path.read_text(encoding="utf-8"))

In [ ]:
remapped_mask = (mask > 0).astype(np.uint8)

print("Original values:", np.unique(mask))
print("Remapped values:", np.unique(remapped_mask))

print("Original label-1 pixels:", np.sum(mask == 1))
print("Original label-2 pixels:", np.sum(mask == 2))
print("Total remapped slick pixels:", np.sum(remapped_mask == 1))
print("Background pixels:", np.sum(remapped_mask == 0))

In [ ]:
import numpy as np
import pandas as pd
from PIL import Image

coverage_rows = []

for _, row in all_splits.iterrows():
    mask_path = dataset_dir / row["mask_path"]

    with Image.open(mask_path) as mask_image:
        mask_array = np.array(mask_image)

    slick_pixels = np.count_nonzero(mask_array > 0)
    total_pixels = mask_array.size

    coverage_rows.append({
        "sample_id": row["sample_id"],
        "slick_pixels": slick_pixels,
        "total_pixels": total_pixels,
        "slick_percentage": 100 * slick_pixels / total_pixels
    })

coverage_df = pd.DataFrame(coverage_rows)

print("Number of masks:", len(coverage_df))
print("Masks with no slick pixels:",
      (coverage_df["slick_pixels"] == 0).sum())

print("\nSlick coverage statistics (%):")
display(coverage_df["slick_percentage"].describe().to_frame())

print("\n5 samples with the lowest slick coverage:")
display(coverage_df.nsmallest(5, "slick_percentage"))

print("\n5 samples with the highest slick coverage:")
display(coverage_df.nlargest(5, "slick_percentage"))

In [ ]:
split_frames = {
    "train": train_df,
    "validation": val_df,
    "test": test_df,
}

split_summary = []

for split_name, split_df in split_frames.items():
    split_coverage = coverage_df[
        coverage_df["sample_id"].isin(split_df["sample_id"])
    ]["slick_percentage"]

    split_summary.append({
        "split": split_name,
        "images": len(split_df),
        "mean_slick_%": split_coverage.mean(),
        "median_slick_%": split_coverage.median(),
        "min_slick_%": split_coverage.min(),
        "max_slick_%": split_coverage.max(),
    })

display(pd.DataFrame(split_summary).round(3))

In [ ]:
from pathlib import Path

ml_dir = dataset_dir / "pipeline_ml"

for folder in [ml_dir / "scripts", ml_dir / "results"]:
    if folder.exists():
        print(f"\n--- {folder.relative_to(dataset_dir)} ---")
        for path in sorted(folder.iterdir()):
            if path.is_file():
                print(path.name)

In [ ]:
model_path = dataset_dir / "pipeline_ml" / "scripts" / "model.py"

print(model_path.read_text(encoding="utf-8"))

In [ ]:
train_path = dataset_dir / "pipeline_ml" / "scripts" / "train.py"

print(train_path.read_text(encoding="utf-8"))

In [ ]:
train_path = dataset_dir / "pipeline_ml" / "scripts" / "train.py"

print(train_path.read_text(encoding="utf-8"))

In [ ]:
config_path = dataset_dir / "pipeline_ml" / "scripts" / "pipeline_ml_config.json"

print(config_path.read_text(encoding="utf-8"))

In [ ]:
from pathlib import Path
import json

config_path = dataset_dir / "pipeline_ml" / "scripts" / "pipeline_ml_config.json"
config = json.loads(config_path.read_text(encoding="utf-8"))

for key in ("dataset_root", "results_dir", "benchmark_dir"):
    resolved = (config_path.parent / config[key]).resolve()
    print(f"{key}: {resolved}")
    print(f"  Exists: {resolved.exists()}")

In [ ]:
train_script = dataset_dir / "pipeline_ml" / "scripts" / "train.py"
train_source = train_script.read_text(encoding="utf-8")

for i, line in enumerate(train_source.splitlines(), start=1):
    if any(term in line for term in [
        "mkdir", "makedirs", "resolve_results_dir",
        "results_dir", "run_dir", "checkpoints"
    ]):
        print(f"{i}: {line}")

In [ ]:
common_path = dataset_dir / "pipeline_ml" / "scripts" / "common.py"
common_source = common_path.read_text(encoding="utf-8").splitlines()

for i, line in enumerate(common_source, start=1):
    if "def ensure_dir" in line:
        print("\n".join(
            f"{j}: {common_source[j-1]}"
            for j in range(i, min(i + 5, len(common_source) + 1))
        ))
        break

In [ ]:
from pathlib import Path
import importlib.util

pipeline_dir = dataset_dir / "pipeline_ml"
scripts_dir = pipeline_dir / "scripts"

print("SPLIT FILES")
for name in ["train.csv", "val.csv", "test.csv"]:
    matches = list(dataset_dir.rglob(name))
    print(f"{name}:")
    for path in matches[:3]:
        print(f"  {path} | Exists: {path.exists()}")

print("\nPYTHON LIBRARIES")
for module in ["torch", "torchvision", "numpy", "PIL", "pandas"]:
    print(f"{module}: {importlib.util.find_spec(module) is not None}")

In [ ]:
common_path = dataset_dir / "pipeline_ml" / "scripts" / "common.py"
common_source = common_path.read_text(encoding="utf-8").splitlines()

for i, line in enumerate(common_source, start=1):
    if "class OilSpillSegmentationDataset" in line:
        end = min(i + 65, len(common_source) + 1)
        print("\n".join(
            f"{j}: {common_source[j-1]}"
            for j in range(i, end)
        ))
        break
else:
    print("Dataset class not found in common.py")

In [ ]:
common_path = dataset_dir / "pipeline_ml" / "scripts" / "common.py"
common_source = common_path.read_text(encoding="utf-8").splitlines()

print("Functions and classes defined in common.py:\n")

for i, line in enumerate(common_source, start=1):
    if line.startswith(("def ", "class ")):
        print(f"{i}: {line}")

In [ ]:
import sys
import torch
from pathlib import Path
from torch.utils.data import DataLoader

# Locate the project scripts
scripts_dir = dataset_dir / "pipeline_ml" / "scripts"

if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# Import the actual functions defined in common.py
from common import load_split_csv, OilSpillSegmentationDataset

# Load the training split and create the dataset
train_samples = load_split_csv(dataset_dir, "train")
train_dataset = OilSpillSegmentationDataset(train_samples)

# Load a small batch of two images
train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=0
)

batch = next(iter(train_loader))

# Display checks
print("Training samples:", len(train_dataset))
print("Image shape:", tuple(batch["image"].shape))
print("Mask shape:", tuple(batch["mask"].shape))
print("Image dtype:", batch["image"].dtype)
print("Mask dtype:", batch["mask"].dtype)
print("Image min/max:", batch["image"].min().item(), batch["image"].max().item())
print("Mask labels:", torch.unique(batch["mask"]).tolist())
print("Slick pixels:", (batch["mask"] == 1).sum().item())

In [ ]:
import sys
import torch
from pathlib import Path
from torch.utils.data import DataLoader

# Locate the project scripts
scripts_dir = dataset_dir / "pipeline_ml" / "scripts"

if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# Import the actual functions defined in common.py
from common import load_split_csv, OilSpillSegmentationDataset

# Load the training split and create the dataset
train_samples = load_split_csv(dataset_dir, "train")
train_dataset = OilSpillSegmentationDataset(train_samples)

# Load a small batch of two images
train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=0
)

batch = next(iter(train_loader))

# Display checks
print("Training samples:", len(train_dataset))
print("Image shape:", tuple(batch["image"].shape))
print("Mask shape:", tuple(batch["mask"].shape))
print("Image dtype:", batch["image"].dtype)
print("Mask dtype:", batch["mask"].dtype)
print("Image min/max:", batch["image"].min().item(), batch["image"].max().item())
print("Mask labels:", torch.unique(batch["mask"]).tolist())
print("Slick pixels:", (batch["mask"] == 1).sum().item())

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(
        torch.cuda.get_device_properties(0).total_memory / (1024**3), 2
    ))
else:
    print("Training will use the CPU unless another supported accelerator is configured.")

In [ ]:
import json

config_path = dataset_dir / "pipeline_ml" / "scripts" / "pipeline_ml_config.json"
config = json.loads(config_path.read_text(encoding="utf-8"))

print("Dataset root:", config["dataset_root"])
print("Results directory:", config["results_dir"])
print("Benchmark directory:", config["benchmark_dir"])

train_path = dataset_dir / "pipeline_ml" / "scripts" / "train.py"
train_lines = train_path.read_text(encoding="utf-8").splitlines()

print("\nTraining defaults:")
for i, line in enumerate(train_lines, start=1):
    if any(term in line for term in [
        'default="train_run"',
        'default=10',
        'default=4',
        'default=1e-3',
        'default="auto"',
        'CrossEntropyLoss',
    ]):
        print(f"{i}: {line}")

In [ ]:
import sys
import subprocess

train_script = dataset_dir / "pipeline_ml" / "scripts" / "train.py"

result = subprocess.run(
    [sys.executable, str(train_script), "--help"],
    capture_output=True,
    text=True,
    timeout=30
)

print("Exit code:", result.returncode)
print(result.stdout)

if result.stderr:
    print("STDERR:")
    print(result.stderr)

### Optional environment modification

This cell installs `tqdm` into the active Python environment. It is retained for provenance but is not required for notebook inspection. Run only if you intentionally want to modify the environment.

In [ ]:
import sys
import subprocess

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "tqdm"],
    capture_output=True,
    text=True
)

print(result.stdout[-4000:])

if result.returncode != 0:
    print("Installation error:")
    print(result.stderr[-3000:])
else:
    print("tqdm installation completed.")

In [ ]:
import sys
import subprocess

train_script = dataset_dir / "pipeline_ml" / "scripts" / "train.py"

result = subprocess.run(
    [sys.executable, str(train_script), "--help"],
    capture_output=True,
    text=True,
    timeout=30
)

print("Exit code:", result.returncode)

if result.stdout:
    print(result.stdout)

if result.stderr:
    print("STDERR:")
    print(result.stderr[-3000:])

In [ ]:
import json

from pathlib import Path

run_dir = (
    dataset_dir
    / "pipeline_ml"
    / "results"
    / "cpu_pilot_1epoch"
)

print("Run directory exists:", run_dir.exists())
print("\nSaved files:")

for path in sorted(run_dir.rglob("*")):
    if path.is_file():
        print(path.relative_to(run_dir))

history_path = run_dir / "history.json"

if history_path.exists():
    print("\nTraining history:")
    print(history_path.read_text(encoding="utf-8"))
else:
    print("\nhistory.json was not found.")

In [ ]:
scripts_dir = dataset_dir / "pipeline_ml" / "scripts"

for filename in ["train.py", "common.py"]:
    path = scripts_dir / filename
    lines = path.read_text(encoding="utf-8").splitlines()

    print(f"\n{'=' * 15} {filename} {'=' * 15}")

    for i, line in enumerate(lines, start=1):
        if any(term in line for term in [
            "metrics_from_confusion",
            "val_iou_per_class",
            "val_mean_iou",
            "mean_iou",
            "confusion",
            "update_confusion",
        ]):
            start = max(1, i - 3)
            end = min(len(lines), i + 5)
            print(f"\n--- lines {start}-{end} ---")
            print("\n".join(
                f"{j}: {lines[j - 1]}"
                for j in range(start, end + 1)
            ))

In [ ]:
test_path = dataset_dir / "pipeline_ml" / "scripts" / "test.py"
test_lines = test_path.read_text(encoding="utf-8").splitlines()

print("\n".join(
    f"{i}: {line}"
    for i, line in enumerate(test_lines, start=1)
))

## 7. Validation, prediction and mask-quality audits

Validation and prediction diagnostics examine model behaviour and mask quality. Global pixel-level metrics and per-image averages are distinct quantities and should not be interchanged.

In [ ]:
import sys
import numpy as np
import torch
from torch.utils.data import DataLoader

scripts_dir = dataset_dir / "pipeline_ml" / "scripts"
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

from common import (
    load_split_csv,
    OilSpillSegmentationDataset,
    get_num_classes,
    update_confusion,
    metrics_from_confusion,
    pick_device,
)
from model import TinyUNet

# Use validation data only
val_samples = load_split_csv(dataset_dir, "val")
val_dataset = OilSpillSegmentationDataset(val_samples)
val_loader = DataLoader(
    val_dataset, batch_size=2, shuffle=False, num_workers=0
)

device = torch.device("cpu")
num_classes = get_num_classes()

checkpoint_path = (
    dataset_dir / "pipeline_ml" / "results"
    / "cpu_pilot_1epoch" / "checkpoints" / "best.pt"
)

model = TinyUNet(num_classes=num_classes).to(device)
checkpoint = torch.load(
    checkpoint_path, map_location=device, weights_only=False
)
model.load_state_dict(checkpoint["model_state"])
model.eval()

conf = np.zeros((num_classes, num_classes), dtype=np.int64)
total_loss = 0.0
criterion = torch.nn.CrossEntropyLoss()

with torch.no_grad():
    for batch in val_loader:
        images = batch["image"].to(device)
        masks = batch["mask"].to(device)

        logits = model(images)
        loss = criterion(logits, masks)
        total_loss += loss.item() * images.size(0)

        predictions = logits.argmax(dim=1).cpu().numpy()
        targets = masks.cpu().numpy()
        update_confusion(conf, predictions, targets, num_classes)

metrics = metrics_from_confusion(conf)

print("Validation samples:", len(val_dataset))
print("Confusion matrix (rows=true, columns=predicted):")
print(conf)
print("\nValidation loss:", round(total_loss / len(val_dataset), 4))
print("Slick IoU:", round(metrics["iou_per_class"][0], 4))
print("All-class mean IoU:", round(metrics["mean_iou_all"], 4))
print("IoU per class [SEA, SLICK]:", metrics["iou_per_class_all"])
print("Pixel accuracy:", round(metrics["pixel_accuracy"], 4))

### Optional training run — baseline

This cell launches a 10-epoch training process and writes run artifacts under the pipeline results directory. It is retained but was not executed during cleanup; do not run unless you intend to start training.

In [ ]:
import subprocess
import sys

scripts_dir = dataset_dir / "pipeline_ml" / "scripts"

cmd = [
    sys.executable,
    str(scripts_dir / "train.py"),
    "--run-name", "cpu_baseline_10epochs",
    "--epochs", "10",
    "--batch-size", "2",
    "--lr", "0.001",
    "--num-workers", "0",
    "--device", "cpu",
    "--seed", "42",
]

result = subprocess.run(
    cmd,
    cwd=str(scripts_dir),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)

print("Exit code:", result.returncode)
print(result.stdout[-12000:])

if result.stderr:
    print("\nMessages / errors:")
    print(result.stderr[-4000:])

In [ ]:
import json
from pathlib import Path

dataset_dir = SEG_DATASET_DIR

history_path = (
    dataset_dir
    / "pipeline_ml"
    / "results"
    / "cpu_baseline_10epochs"
    / "history.json"
)

with open(history_path, "r", encoding="utf-8") as f:
    history_data = json.load(f)

epochs = history_data["history"]

print("Epoch | Train loss | Val loss | Slick IoU")
print("-" * 46)

for row in epochs:
    print(
        f'{row["epoch"]:>5} | '
        f'{row["train_loss"]:>10.4f} | '
        f'{row["val_loss"]:>8.4f} | '
        f'{row["val_mean_iou"]:>9.4f}'
    )

best_epoch = max(epochs, key=lambda row: row["val_mean_iou"])

print("\nBest epoch:", best_epoch["epoch"])
print("Best validation slick IoU:", round(best_epoch["val_mean_iou"], 4))
print("Saved best score:", round(history_data["best_miou"], 4))

In [ ]:
import sys
import numpy as np
import torch
from torch.utils.data import DataLoader

scripts_dir = dataset_dir / "pipeline_ml" / "scripts"
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

from common import (
    load_split_csv,
    OilSpillSegmentationDataset,
    get_num_classes,
    update_confusion,
    metrics_from_confusion,
)
from model import TinyUNet

val_samples = load_split_csv(dataset_dir, "val")
val_dataset = OilSpillSegmentationDataset(val_samples)
val_loader = DataLoader(
    val_dataset, batch_size=2, shuffle=False, num_workers=0
)

device = torch.device("cpu")
num_classes = get_num_classes()

checkpoint_path = (
    dataset_dir / "pipeline_ml" / "results"
    / "cpu_baseline_10epochs" / "checkpoints" / "best.pt"
)

model = TinyUNet(num_classes=num_classes).to(device)
checkpoint = torch.load(
    checkpoint_path, map_location=device, weights_only=False
)
model.load_state_dict(checkpoint["model_state"])
model.eval()

conf = np.zeros((num_classes, num_classes), dtype=np.int64)

with torch.no_grad():
    for batch in val_loader:
        images = batch["image"].to(device)
        masks = batch["mask"].to(device)
        predictions = model(images).argmax(dim=1).cpu().numpy()
        targets = masks.cpu().numpy()
        update_confusion(conf, predictions, targets, num_classes)

metrics = metrics_from_confusion(conf)
tp = conf[1, 1]
fn = conf[1, 0]
fp = conf[0, 1]

print("Confusion matrix (rows=true, columns=predicted):")
print(conf)
print("\nSlick IoU:", round(metrics["iou_per_class"][0], 4))
print("Slick recall:", round(tp / (tp + fn + 1e-9), 4))
print("Slick precision:", round(tp / (tp + fp + 1e-9), 4))
print("Pixel accuracy:", round(metrics["pixel_accuracy"], 4))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

model.eval()

# Show the first 3 validation examples
samples_to_show = min(3, len(val_dataset))

fig, axes = plt.subplots(samples_to_show, 3, figsize=(15, 5 * samples_to_show))

if samples_to_show == 1:
    axes = np.expand_dims(axes, axis=0)

with torch.no_grad():
    for i in range(samples_to_show):
        sample = val_dataset[i]
        image = sample["image"].unsqueeze(0).to(device)
        true_mask = sample["mask"].cpu().numpy()
        predicted_mask = model(image).argmax(dim=1)[0].cpu().numpy()

        axes[i, 0].imshow(image[0].cpu().permute(1, 2, 0).numpy())
        axes[i, 0].set_title("Input SAR image")

        axes[i, 1].imshow(true_mask, cmap="gray", vmin=0, vmax=1)
        axes[i, 1].set_title("Ground-truth mask")

        axes[i, 2].imshow(predicted_mask, cmap="gray", vmin=0, vmax=1)
        axes[i, 2].set_title("Predicted mask")

        for ax in axes[i]:
            ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import torch

model.eval()
image_results = []

with torch.no_grad():
    for i in range(len(val_dataset)):
        sample = val_dataset[i]
        image = sample["image"].unsqueeze(0).to(device)
        true_mask = sample["mask"].cpu().numpy()

        pred_mask = (
            model(image)
            .argmax(dim=1)[0]
            .cpu()
            .numpy()
        )

        true_slick = true_mask == 1
        pred_slick = pred_mask == 1

        true_pct = 100 * true_slick.mean()
        pred_pct = 100 * pred_slick.mean()

        tp = np.logical_and(true_slick, pred_slick).sum()
        fp = np.logical_and(~true_slick, pred_slick).sum()
        fn = np.logical_and(true_slick, ~pred_slick).sum()

        precision = tp / (tp + fp + 1e-9)
        recall = tp / (tp + fn + 1e-9)
        iou = tp / (tp + fp + fn + 1e-9)

        image_results.append({
            "index": i,
            "true_slick_pct": true_pct,
            "pred_slick_pct": pred_pct,
            "extra_pct_points": pred_pct - true_pct,
            "precision": precision,
            "recall": recall,
            "iou": iou,
        })

# Show images with the greatest excess predicted oil area
image_results.sort(
    key=lambda row: row["extra_pct_points"],
    reverse=True,
)

print("Worst validation overpredictions")
print("Idx | True oil % | Pred oil % | Excess points | Precision | Recall | IoU")
print("-" * 79)

for row in image_results[:10]:
    print(
        f'{row["index"]:>3} | '
        f'{row["true_slick_pct"]:>10.3f} | '
        f'{row["pred_slick_pct"]:>10.3f} | '
        f'{row["extra_pct_points"]:>13.3f} | '
        f'{row["precision"]:>9.3f} | '
        f'{row["recall"]:>6.3f} | '
        f'{row["iou"]:>5.3f}'
    )

In [ ]:
import numpy as np
import torch

model.eval()

thresholds = [0.30, 0.40, 0.50, 0.60, 0.70, 0.80]
threshold_results = []

with torch.no_grad():
    for threshold in thresholds:
        tp = fp = fn = 0

        for i in range(len(val_dataset)):
            sample = val_dataset[i]
            image = sample["image"].unsqueeze(0).to(device)
            true_mask = sample["mask"].cpu().numpy() == 1

            logits = model(image)
            oil_probability = torch.softmax(logits, dim=1)[0, 1].cpu().numpy()
            pred_mask = oil_probability >= threshold

            tp += np.logical_and(pred_mask, true_mask).sum()
            fp += np.logical_and(pred_mask, ~true_mask).sum()
            fn += np.logical_and(~pred_mask, true_mask).sum()

        precision = tp / (tp + fp + 1e-9)
        recall = tp / (tp + fn + 1e-9)
        iou = tp / (tp + fp + fn + 1e-9)

        threshold_results.append(
            (threshold, precision, recall, iou)
        )

print("Threshold | Precision | Recall | Slick IoU")
print("-" * 43)

for threshold, precision, recall, iou in threshold_results:
    print(
        f"{threshold:>9.2f} | "
        f"{precision:>9.4f} | "
        f"{recall:>6.4f} | "
        f"{iou:>9.4f}"
    )

In [ ]:
import inspect

print(inspect.signature(OilSpillSegmentationDataset))
print(inspect.getsource(OilSpillSegmentationDataset.__init__))

In [ ]:
val_samples = load_split_csv(dataset_dir, "val")
val_dataset = OilSpillSegmentationDataset(val_samples)

print("Validation samples:", len(val_dataset))

In [ ]:
checkpoint_path = (
    dataset_dir
    / "pipeline_ml"
    / "results"
    / "cpu_baseline_10epochs"
    / "checkpoints"
    / "best.pt"
)

model = TinyUNet(num_classes=2).to(device)

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state"])
model.eval()

print("Checkpoint exists:", checkpoint_path.exists())
print("Best validation slick IoU:", checkpoint.get("best_miou"))
print("Model ready for visualization.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

indices = [1, 2, 10]
threshold = 0.50

model.eval()

fig, axes = plt.subplots(len(indices), 4, figsize=(18, 4 * len(indices)))

with torch.no_grad():
    for row, idx in enumerate(indices):
        sample = val_dataset[idx]

        image = sample["image"].unsqueeze(0).to(device)
        true_mask = sample["mask"].cpu().numpy()

        probabilities = torch.softmax(model(image), dim=1)[0, 1].cpu().numpy()
        pred_mask = probabilities >= threshold

        original = sample["image"].cpu().numpy().transpose(1, 2, 0)

        axes[row, 0].imshow(original)
        axes[row, 0].set_title(f"Validation index {idx}: Original")

        axes[row, 1].imshow(true_mask, cmap="gray", vmin=0, vmax=1)
        axes[row, 1].set_title("Ground-truth mask")

        axes[row, 2].imshow(pred_mask, cmap="gray", vmin=0, vmax=1)
        axes[row, 2].set_title("Predicted oil mask (0.50)")

        axes[row, 3].imshow(original)
        axes[row, 3].imshow(pred_mask, cmap="Reds", alpha=0.45, vmin=0, vmax=1)
        axes[row, 3].set_title("Prediction overlay")

        for col in range(4):
            axes[row, col].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import torch
import numpy as np

indices = [1, 2, 10]
fig, axes = plt.subplots(len(indices), 3, figsize=(15, 12))

model.eval()

with torch.no_grad():
    for row, idx in enumerate(indices):
        sample = val_dataset[idx]
        image = sample["image"].unsqueeze(0).to(device)
        true_mask = sample["mask"].cpu().numpy()

        probs = torch.softmax(model(image), dim=1)[0, 1].cpu().numpy()

        axes[row, 0].imshow(
            sample["image"].cpu().numpy().transpose(1, 2, 0)
        )
        axes[row, 0].set_title(f"Original — index {idx}")

        axes[row, 1].imshow(true_mask, cmap="gray", vmin=0, vmax=1)
        axes[row, 1].set_title("Ground truth")

        im = axes[row, 2].imshow(
            probs, cmap="magma", vmin=0, vmax=1
        )
        axes[row, 2].set_title("Predicted oil probability")

        for ax in axes[row]:
            ax.axis("off")

fig.colorbar(im, ax=axes[:, 2], shrink=0.7, label="Oil probability")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

indices = [1, 2, 10]

fig, axes = plt.subplots(len(indices), 3, figsize=(13, 10))

for row, idx in enumerate(indices):
    sample = val_dataset[idx]

    image = sample["image"].cpu().numpy().transpose(1, 2, 0)
    mask = sample["mask"].cpu().numpy()

    # Brightness in the original image
    brightness = image.mean(axis=2)

    axes[row, 0].imshow(brightness, cmap="gray")
    axes[row, 0].set_title(f"Image brightness — {idx}")

    axes[row, 1].imshow(mask, cmap="gray", vmin=0, vmax=1)
    axes[row, 1].set_title("Remapped mask")

    axes[row, 2].imshow(brightness, cmap="gray")
    axes[row, 2].imshow(mask == 1, cmap="Reds", alpha=0.6)
    axes[row, 2].set_title("Ground-truth overlay")

    for ax in axes[row]:
        ax.axis("off")

plt.tight_layout()
plt.show()

sample = val_dataset[10]
image = sample["image"].cpu().numpy()
mask = sample["mask"].cpu().numpy()

print("Image shape:", image.shape)
print("Mask shape:", mask.shape)
print("Image min/max:", image.min(), image.max())
print("Mask labels:", np.unique(mask))
print("Dark pixels in index 10:", np.mean(image.mean(axis=0) < 0.02))

In [ ]:
import inspect
from common import OilSpillSegmentationDataset

print(inspect.getsource(OilSpillSegmentationDataset))

In [ ]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

sample = val_samples[10]

print("Sample ID:", sample.sample_id)
print("Image path:", sample.image_path)
print("Mask path:", sample.mask_path)

original_image = np.asarray(
    Image.open(sample.image_path).convert("RGB")
)
original_mask = np.asarray(
    Image.open(sample.mask_path).convert("L")
)

print("Original image shape:", original_image.shape)
print("Original mask shape:", original_mask.shape)
print("Original mask values:", np.unique(original_mask))
print("Fraction of exactly black RGB pixels:",
      np.mean(np.all(original_image == 0, axis=2)))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(original_image)
axes[0].set_title("Original RGB image")

axes[1].imshow(original_mask, cmap="gray")
axes[1].set_title("Original mask")

axes[2].imshow(original_image)
axes[2].imshow(original_mask > 0, cmap="Reds", alpha=0.5)
axes[2].set_title("Original image + mask overlay")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

sample = val_samples[10]

image = np.asarray(
    Image.open(sample.image_path).convert("RGB")
)
mask = np.asarray(
    Image.open(sample.mask_path).convert("L")
)

# The white diagonal region is near the bottom of the image.
# Inspect the bottom 100 rows separately.
bottom_image = image[500:, :, :]
bottom_mask = mask[500:, :]

print("Bottom-region mask values:", np.unique(bottom_mask))
print("Bottom-region slick pixels:", np.sum(bottom_mask > 0))
print("Bottom-region total pixels:", bottom_mask.size)
print("Bottom-region slick percentage:",
      100 * np.mean(bottom_mask > 0))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].imshow(bottom_image)
axes[0].set_title("Original image — bottom 100 rows")

axes[1].imshow(bottom_mask, cmap="gray", vmin=0, vmax=1)
axes[1].set_title("Ground-truth mask — bottom 100 rows")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
from PIL import Image

sample = val_samples[10]

image = np.asarray(Image.open(sample.image_path).convert("RGB"))
mask = np.asarray(Image.open(sample.mask_path).convert("L"))

# Pixels that are nearly white in all three channels
white_region = np.all(image >= 245, axis=2)
oil_region = mask > 0

print("Nearly-white pixels:", int(white_region.sum()))
print("Nearly-white area (%):", round(100 * white_region.mean(), 4))
print("Nearly-white pixels labeled oil:", int((white_region & oil_region).sum()))
print("Nearly-white pixels labeled sea:", int((white_region & ~oil_region).sum()))

In [ ]:
import numpy as np
from PIL import Image

splits = {
    "train": load_split_csv(dataset_dir, "train"),
    "val": load_split_csv(dataset_dir, "val"),
    "test": load_split_csv(dataset_dir, "test"),
}

for split_name, samples in splits.items():
    affected = 0
    total_images = len(samples)
    white_pixel_counts = []

    for sample in samples:
        image = np.asarray(
            Image.open(sample.image_path).convert("RGB")
        )
        mask = np.asarray(
            Image.open(sample.mask_path).convert("L")
        )

        white = np.all(image >= 245, axis=2)
        white_pixel_counts.append(100 * white.mean())

        if white.mean() > 0.01:
            affected += 1

    print(f"\n{split_name.upper()}")
    print(f"Images: {total_images}")
    print(f"Images with >1% nearly-white pixels: {affected}")
    print(f"Mean nearly-white area: {np.mean(white_pixel_counts):.2f}%")
    print(f"Maximum nearly-white area: {np.max(white_pixel_counts):.2f}%")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

audit_results = []

for split_name in ["train", "val"]:
    for sample in splits[split_name]:
        image = np.asarray(
            Image.open(sample.image_path).convert("RGB")
        )
        white = np.all(image >= 245, axis=2)

        audit_results.append({
            "split": split_name,
            "sample_id": sample.sample_id,
            "path": sample.image_path,
            "white_pct": 100 * white.mean(),
        })

audit_results.sort(
    key=lambda row: row["white_pct"],
    reverse=True,
)

print("Split | Sample ID | Nearly-white area")
print("-" * 65)

for row in audit_results[:10]:
    print(
        f'{row["split"]:>5} | '
        f'{row["sample_id"]:<35} | '
        f'{row["white_pct"]:>7.2f}%'
    )

fig, axes = plt.subplots(2, 3, figsize=(15, 7))

for ax, row in zip(axes.flat, audit_results[:6]):
    image = np.asarray(
        Image.open(row["path"]).convert("RGB")
    )
    ax.imshow(image)
    ax.set_title(
        f'{row["split"]}: {row["sample_id"]}\n'
        f'White area: {row["white_pct"]:.1f}%'
    )
    ax.axis("off")

for ax in axes.flat[len(audit_results[:6]):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Inspect the three images with the largest white regions
top_samples = [
    row for row in audit_results[:3]
]

fig, axes = plt.subplots(3, 3, figsize=(15, 12))

for i, row in enumerate(top_samples):
    image = np.asarray(
        Image.open(row["path"]).convert("RGB")
    )

    sample = next(
        s for s in splits[row["split"]]
        if s.sample_id == row["sample_id"]
    )

    mask = np.asarray(
        Image.open(sample.mask_path).convert("L")
    )

    nearly_white = np.all(image >= 245, axis=2)
    oil = mask > 0

    axes[i, 0].imshow(image)
    axes[i, 0].set_title(row["sample_id"])

    axes[i, 1].imshow(mask, cmap="gray")
    axes[i, 1].set_title("Original ground-truth mask")

    axes[i, 2].imshow(image)
    axes[i, 2].imshow(oil, cmap="Reds", alpha=0.5)
    axes[i, 2].set_title(
        f"Overlay | oil in white area: "
        f"{np.sum(nearly_white & oil)} pixels"
    )

    print(
        row["sample_id"],
        "| white area:",
        f'{row["white_pct"]:.2f}%',
        "| oil pixels inside white area:",
        int(np.sum(nearly_white & oil)),
        "| total oil pixels:",
        int(oil.sum())
    )

for ax in axes.flat:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

sample = next(
    s for s in splits["val"]
    if s.sample_id == "IMG_19_TILE_019"
)

image = np.asarray(
    Image.open(sample.image_path).convert("RGB"),
    dtype=np.float32
) / 255.0

brightness = image.mean(axis=2)
nearly_white = np.all(image >= 245, axis=2)

# Fraction of nearly-white pixels in each row and column
white_by_row = nearly_white.mean(axis=1)
white_by_col = nearly_white.mean(axis=0)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].imshow(image)
axes[0].set_title("Original image")

axes[1].plot(white_by_row, np.arange(len(white_by_row)))
axes[1].invert_yaxis()
axes[1].set_xlabel("White-pixel fraction")
axes[1].set_ylabel("Row")
axes[1].set_title("White fraction by row")

axes[2].plot(np.arange(len(white_by_col)), white_by_col)
axes[2].set_xlabel("Column")
axes[2].set_ylabel("White-pixel fraction")
axes[2].set_title("White fraction by column")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

sample = next(
    s for s in splits["val"]
    if s.sample_id == "IMG_19_TILE_019"
)

image = np.asarray(
    Image.open(sample.image_path).convert("RGB"),
    dtype=np.float32
) / 255.0

brightness = image.mean(axis=2)

print("Brightness min:", brightness.min())
print("Brightness max:", brightness.max())
print("Brightness percentiles:",
      np.percentile(brightness, [10, 25, 50, 75, 90, 95, 99]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].imshow(brightness, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Grayscale brightness")

axes[1].hist(brightness.ravel(), bins=50)
axes[1].set_xlabel("Mean RGB brightness")
axes[1].set_ylabel("Pixel count")
axes[1].set_title("Brightness distribution")

for ax in axes:
    ax.set_axis_off() if ax is axes[0] else None

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

sample = next(
    s for s in splits["val"]
    if s.sample_id == "IMG_19_TILE_019"
)

image = np.asarray(
    Image.open(sample.image_path).convert("RGB"),
    dtype=np.float32
) / 255.0

brightness = image.mean(axis=2)

# Compare several candidate brightness thresholds
thresholds = [0.80, 0.85, 0.90, 0.95]

fig, axes = plt.subplots(1, len(thresholds) + 1, figsize=(18, 4))

axes[0].imshow(image)
axes[0].set_title("Original")
axes[0].axis("off")

for ax, threshold in zip(axes[1:], thresholds):
    candidate = brightness >= threshold
    ax.imshow(candidate, cmap="gray", vmin=0, vmax=1)
    ax.set_title(
        f"Threshold {threshold}\n"
        f"{100 * candidate.mean():.1f}% marked"
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
from PIL import Image

sample = next(
    s for s in splits["val"]
    if s.sample_id == "IMG_19_TILE_019"
)

image = np.asarray(
    Image.open(sample.image_path).convert("RGB"),
    dtype=np.float32
) / 255.0

mask = np.asarray(
    Image.open(sample.mask_path).convert("L")
)

brightness = image.mean(axis=2)

# Candidate background region
background = brightness >= 0.80
oil = mask > 0

print("Candidate background area:",
      f"{100 * background.mean():.2f}%")
print("Oil pixels in candidate background:",
      int(np.sum(background & oil)))
print("Total oil pixels:",
      int(oil.sum()))
print("Oil pixels outside candidate background:",
      int(np.sum(~background & oil)))

In [ ]:
import numpy as np
from PIL import Image

print(
    "Sample ID".ljust(24),
    "Background %".rjust(13),
    "Oil in background".rjust(18),
    "Total oil".rjust(12)
)
print("-" * 70)

for row in audit_results[:6]:
    sample = next(
        s for s in splits[row["split"]]
        if s.sample_id == row["sample_id"]
    )

    image = np.asarray(
        Image.open(sample.image_path).convert("RGB"),
        dtype=np.float32
    ) / 255.0

    mask = np.asarray(
        Image.open(sample.mask_path).convert("L")
    )

    candidate_background = image.mean(axis=2) >= 0.80
    oil = mask > 0

    print(
        f'{sample.sample_id:<24}'
        f'{100 * candidate_background.mean():>12.2f}%'
        f'{np.sum(candidate_background & oil):>18}'
        f'{oil.sum():>12}'
    )

In [ ]:
import numpy as np
from PIL import Image

for split_name in ["train", "val"]:
    samples = splits[split_name]

    total_pixels = 0
    background_pixels = 0
    oil_pixels_in_background = 0
    total_oil_pixels = 0
    affected_samples = 0

    for sample in samples:
        image = np.asarray(
            Image.open(sample.image_path).convert("RGB"),
            dtype=np.float32
        ) / 255.0

        mask = np.asarray(
            Image.open(sample.mask_path).convert("L")
        )

        candidate_background = image.mean(axis=2) >= 0.80
        oil = mask > 0

        total_pixels += candidate_background.size
        background_pixels += candidate_background.sum()
        oil_pixels_in_background += np.sum(
            candidate_background & oil
        )
        total_oil_pixels += oil.sum()

        if candidate_background.mean() > 0.01:
            affected_samples += 1

    print(f"\n{split_name.upper()}")
    print("Samples:", len(samples))
    print("Samples with >1% candidate background:", affected_samples)
    print(
        "Candidate background area:",
        f"{100 * background_pixels / total_pixels:.2f}%"
    )
    print("Oil pixels inside candidate background:",
          int(oil_pixels_in_background))
    print("Total oil pixels:", int(total_oil_pixels))
    print(
        "Fraction of oil labels inside candidate background:",
        f"{100 * oil_pixels_in_background / max(total_oil_pixels, 1):.4f}%"
    )

In [ ]:
import numpy as np
import torch
from PIL import Image

model.eval()

tp_all = fp_all = fn_all = 0
tp_valid = fp_valid = fn_valid = 0

with torch.no_grad():
    for sample in val_samples:
        image_np = np.asarray(
            Image.open(sample.image_path).convert("RGB"),
            dtype=np.float32
        ) / 255.0

        original_mask = np.asarray(
            Image.open(sample.mask_path).convert("L")
        )

        image_tensor = (
            torch.from_numpy(image_np)
            .permute(2, 0, 1)
            .unsqueeze(0)
            .to(device)
        )

        true_oil = original_mask > 0

        oil_probability = torch.softmax(
            model(image_tensor), dim=1
        )[0, 1].cpu().numpy()

        pred_oil = oil_probability >= 0.50

        # Metrics on all pixels
        tp_all += np.sum(pred_oil & true_oil)
        fp_all += np.sum(pred_oil & ~true_oil)
        fn_all += np.sum(~pred_oil & true_oil)

        # Experimental validity mask
        valid = image_np.mean(axis=2) < 0.80

        tp_valid += np.sum(pred_oil & true_oil & valid)
        fp_valid += np.sum(pred_oil & ~true_oil & valid)
        fn_valid += np.sum(~pred_oil & true_oil & valid)

def print_metrics(name, tp, fp, fn):
    precision = tp / (tp + fp + 1e-9)
    recall = tp / (tp + fn + 1e-9)
    iou = tp / (tp + fp + fn + 1e-9)

    print(f"\n{name}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"Slick IoU: {iou:.4f}")

print_metrics("All validation pixels", tp_all, fp_all, fn_all)
print_metrics("Candidate-valid pixels only", tp_valid, fp_valid, fn_valid)

## 8. Training-script and configuration investigations

Inspect training code, command-line arguments and pipeline configuration. Cells that invoke training or modify scripts are retained as experiments; review their effects before running them.

In [ ]:
from pathlib import Path

train_script = scripts_dir / "train.py"
source = train_script.read_text(encoding="utf-8")

for line_number, line in enumerate(source.splitlines(), start=1):
    if any(term in line.lower() for term in [
        "crossentropyloss",
        "class_weight",
        "weight=",
        "argparse",
        "loss_fn",
    ]):
        print(f"{line_number}: {line}")

In [ ]:
source_lines = train_script.read_text(encoding="utf-8").splitlines()

for start, end in [(20, 45), (85, 115)]:
    print(f"\n--- train.py lines {start}-{end} ---")
    for line_number in range(start, min(end, len(source_lines)) + 1):
        print(f"{line_number}: {source_lines[line_number - 1]}")

In [ ]:
for line_number in range(108, min(145, len(source_lines)) + 1):
    print(f"{line_number}: {source_lines[line_number - 1]}")

### Optional source-code modification

This cell edits the dataset's `pipeline_ml/scripts/train.py` after creating a backup. It changes a training argument/loss configuration. Review the diff and backup before running; cleanup did not execute it or modify project scripts.

In [ ]:
from pathlib import Path

train_script = scripts_dir / "train.py"
source = train_script.read_text(encoding="utf-8")

backup_path = train_script.with_name("train.py.before_class_weight.bak")

if not backup_path.exists():
    backup_path.write_text(source, encoding="utf-8")
    print("Backup created:", backup_path)
else:
    print("Backup already exists:", backup_path)

# Avoid accidentally applying the edit twice.
if "--slick-weight" in source:
    print("The --slick-weight option already exists. No changes made.")
else:
    old_arg_block = '    p.add_argument("--seed", type=int, default=42)'
    new_arg_block = old_arg_block + '\n    p.add_argument("--slick-weight", type=float, default=1.0)'

    old_criterion = '    criterion = nn.CrossEntropyLoss()\n\n    history = []'
    new_criterion = '''    class_weights = torch.tensor(
        [1.0, args.slick_weight], dtype=torch.float32, device=device
    )
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    history = []'''

    if old_arg_block not in source:
        raise ValueError("Could not find the expected --seed argument; no edit applied.")
    if old_criterion not in source:
        raise ValueError("Could not find the expected training criterion; no edit applied.")

    updated = source.replace(old_arg_block, new_arg_block, 1)
    updated = updated.replace(old_criterion, new_criterion, 1)

    # Verify only the intended training criterion changed.
    if updated.count("criterion = nn.CrossEntropyLoss()") != 1:
        raise ValueError("Unexpected validation criterion count; no edit applied.")

    train_script.write_text(updated, encoding="utf-8")
    print("Updated:", train_script)

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    str(scripts_dir / "train.py"),
    "--help",
]

result = subprocess.run(
    cmd,
    cwd=str(scripts_dir),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)

print("Exit code:", result.returncode)
print(result.stdout)

if result.stderr:
    print("STDERR:")
    print(result.stderr)

### Optional training run — weighted experiment

This cell launches a 10-epoch weighted training experiment. It is retained as research history and was not executed during cleanup.

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    str(scripts_dir / "train.py"),
    "--run-name", "cpu_weighted_3",
    "--epochs", "10",
    "--batch-size", "2",
    "--lr", "0.001",
    "--num-workers", "0",
    "--device", "cpu",
    "--seed", "42",
    "--slick-weight", "3.0",
]

result = subprocess.run(
    cmd,
    cwd=str(scripts_dir),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)

print("Exit code:", result.returncode)
print(result.stdout[-10000:])

if result.stderr:
    print("\nSTDERR (last 3000 characters):")
    print(result.stderr[-3000:])

In [ ]:
import json
from pathlib import Path

runs = {
    "Baseline (weight 1.0)": dataset_dir / "pipeline_ml" / "results" / "cpu_baseline_10epochs",
    "Weighted (weight 3.0)": dataset_dir / "pipeline_ml" / "results" / "cpu_weighted_3",
}

for name, run_dir in runs.items():
    history_path = run_dir / "checkpoints" / "history.json"

    if not history_path.exists():
        history_path = run_dir / "history.json"

    with open(history_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    history = data["history"]
    best = max(history, key=lambda row: row["val_mean_iou"])

    print(f"\n{name}")
    print("Best epoch:", best["epoch"])
    print("Best slick IoU:", round(best["val_mean_iou"], 4))
    print("Validation loss:", round(best["val_loss"], 4))

In [ ]:
import torch
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader

from model import TinyUNet
from common import (
    load_split_csv,
    OilSpillSegmentationDataset,
    get_num_classes,
    update_confusion,
    metrics_from_confusion,
)

val_samples = load_split_csv(dataset_dir, "val")
val_dataset = OilSpillSegmentationDataset(val_samples)
val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=0,
)

device = torch.device("cpu")
num_classes = get_num_classes()

runs_to_compare = {
    "Baseline (weight 1.0)": (
        dataset_dir / "pipeline_ml" / "results"
        / "cpu_baseline_10epochs" / "checkpoints" / "best.pt"
    ),
    "Weighted (weight 3.0)": (
        dataset_dir / "pipeline_ml" / "results"
        / "cpu_weighted_3" / "checkpoints" / "best.pt"
    ),
}

comparison_rows = []

for run_name, checkpoint_path in runs_to_compare.items():
    model = TinyUNet(num_classes=num_classes).to(device)
    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=False,
    )
    model.load_state_dict(checkpoint["model_state"])
    model.eval()

    conf = np.zeros((num_classes, num_classes), dtype=np.int64)

    with torch.no_grad():
        for batch in val_loader:
            images = batch["image"].to(device)
            masks = batch["mask"].to(device)
            predictions = model(images).argmax(dim=1)

            update_confusion(
                conf,
                masks.cpu().numpy(),
                predictions.cpu().numpy(),
                num_classes,
            )

    metrics = metrics_from_confusion(conf)

    # Binary confusion matrix: rows=true, columns=predicted.
    tn, fp, fn, tp = conf.ravel()
    precision = tp / (tp + fp + 1e-9)
    recall = tp / (tp + fn + 1e-9)

    comparison_rows.append({
        "Run": run_name,
        "Slick IoU": metrics["mean_iou"],
        "Precision": precision,
        "Recall": recall,
        "False positives": int(fp),
        "Missed slick pixels": int(fn),
    })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df.round(4))

In [ ]:
# Inspect the model architecture
model_source = (scripts_dir / "model.py").read_text(encoding="utf-8")

print("===== model.py =====")
print(model_source)

# Inspect the training arguments
train_source = (scripts_dir / "train.py").read_text(encoding="utf-8")
train_lines = train_source.splitlines()

print("\n===== Training arguments =====")
for i, line in enumerate(train_lines, start=1):
    if 50 <= i <= 80:
        print(f"{i}: {line}")

## 9. Metrics, checkpoints and weighted CE + Dice experiments

Retain metric-function investigations, confusion-matrix examples, checkpoint/history inspection and weighted cross-entropy plus Dice experiments. The original model/loss definitions are preserved.

In [ ]:
common_source = (scripts_dir / "common.py").read_text(encoding="utf-8")
common_lines = common_source.splitlines()

for i, line in enumerate(common_lines, start=1):
    if "def update_confusion" in line or "def metrics_from_confusion" in line:
        start = max(1, i - 2)
        end = min(len(common_lines), i + 35)

        print(f"\n===== common.py lines {start}-{end} =====")
        for j in range(start, end + 1):
            print(f"{j}: {common_lines[j - 1]}")
            

In [ ]:
common_source = (scripts_dir / "common.py").read_text(encoding="utf-8")
common_lines = common_source.splitlines()

for i, line in enumerate(common_lines, start=1):
    if "def update_confusion" in line or "def metrics_from_confusion" in line:
        start = max(1, i - 2)
        end = min(len(common_lines), i + 35)

        print(f"\n===== common.py lines {start}-{end} =====")
        for j in range(start, end + 1):
            print(f"{j}: {common_lines[j - 1]}")

In [ ]:

def metrics_from_confusion(conf: np.ndarray) -> dict:
    eps = 1e-9

    # Rows = actual classes; columns = predicted classes.
    tp = np.diag(conf).astype(np.float64)
    fp = conf.sum(axis=0) - tp
    fn = conf.sum(axis=1) - tp

    # Per-class metrics.
    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    f1 = 2 * precision * recall / (precision + recall + eps)
    iou = tp / (tp + fp + fn + eps)

    accuracy = tp.sum() / (conf.sum() + eps)

    # Class 1 = oil slick.
    target_class_ids = TARGET_CLASS_IDS

    return {
        "pixel_accuracy": float(accuracy),
        "precision": float(np.mean(precision[target_class_ids])),
        "recall": float(np.mean(recall[target_class_ids])),
        "f1_score": float(np.mean(f1[target_class_ids])),
        "mean_iou": float(np.mean(iou[target_class_ids])),
        "iou_per_class": [float(iou[c]) for c in target_class_ids],
        "target_class_ids": target_class_ids,
        "precision_per_class": [float(precision[c]) for c in target_class_ids],
        "recall_per_class": [float(recall[c]) for c in target_class_ids],
        "f1_per_class": [float(f1[c]) for c in target_class_ids],
        "mean_iou_all": float(np.mean(iou)),
        "iou_per_class_all": [float(v) for v in iou.tolist()],
    }


In [ ]:
import importlib
import common

importlib.reload(common)

print(common.__file__)
print(common.metrics_from_confusion.__code__.co_consts)

In [ ]:
import numpy as np

example_confusion = np.array([
    [90, 10],
    [20, 30],
])

metrics = common.metrics_from_confusion(example_confusion)

print(metrics)

In [ ]:

def metrics_from_confusion(conf: np.ndarray) -> dict:
    eps = 1e-9

    # Rows = actual class; columns = predicted class.
    tp = np.diag(conf).astype(np.float64)
    fp = conf.sum(axis=0) - tp
    fn = conf.sum(axis=1) - tp

    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    f1 = 2 * precision * recall / (precision + recall + eps)
    iou = tp / (tp + fp + fn + eps)

    accuracy = tp.sum() / (conf.sum() + eps)

    target_class_ids = TARGET_CLASS_IDS

    return {
        "pixel_accuracy": float(accuracy),
        "precision": float(np.mean(precision[target_class_ids])),
        "recall": float(np.mean(recall[target_class_ids])),
        "f1_score": float(np.mean(f1[target_class_ids])),
        "mean_iou": float(np.mean(iou[target_class_ids])),
        "iou_per_class": [float(iou[c]) for c in target_class_ids],
        "target_class_ids": target_class_ids,
        "precision_per_class": [
            float(precision[c]) for c in target_class_ids
        ],
        "recall_per_class": [
            float(recall[c]) for c in target_class_ids
        ],
        "f1_per_class": [
            float(f1[c]) for c in target_class_ids
        ],
        "mean_iou_all": float(np.mean(iou)),
        "iou_per_class_all": [float(v) for v in iou.tolist()],
    }


### Portable project-root setup for pipeline experiments

These later cells revisit the same segmentation project and its pipeline results. They now reuse the central project configuration rather than embedding a machine-specific Windows path.

In [ ]:
from pathlib import Path
import sys

project_dir = PROJECT_ROOT

dataset_dir = project_dir / "oilspill-detection-dataset_20260912-1b9a62c"
scripts_dir = dataset_dir / "pipeline_ml" / "scripts"

if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

print("Scripts directory exists:", scripts_dir.exists())
print("common.py exists:", (scripts_dir / "common.py").exists())
print("Python:", sys.executable)

In [ ]:
import importlib
import common

importlib.reload(common)

print("Imported from:", common.__file__)

In [ ]:
import inspect

print(inspect.getsource(common.metrics_from_confusion))

In [ ]:
import numpy as np

example_confusion = np.array([
    [90, 10],
    [20, 30],
])

metrics = common.metrics_from_confusion(example_confusion)

for name in ["precision", "recall", "f1_score", "mean_iou"]:
    print(f"{name}: {metrics[name]:.4f}")

In [ ]:
from pathlib import Path

project_root = PROJECT_ROOT

dataset_dir = (
    project_root / "oilspill-detection-dataset_20260912-1b9a62c"
)

scripts_dir = dataset_dir / "pipeline_ml" / "scripts"

results_dir = dataset_dir / "pipeline_ml" / "results"

print("Project root:", project_root)
print("Dataset exists:", dataset_dir.exists())
print("Scripts folder exists:", scripts_dir.exists())
print("Results folder exists:", results_dir.exists())

In [ ]:
run_dir = results_dir / "cpu_lr3e4_20epochs"

history_path = run_dir / "history.json"
checkpoint_dir = run_dir / "checkpoints"

print("Experiment folder exists:", run_dir.exists())
print("History exists:", history_path.exists())
print("Checkpoints folder exists:", checkpoint_dir.exists())

In [ ]:
import json
import pandas as pd

with open(history_path, "r") as f:
    history = json.load(f)

print("History type:", type(history))

if isinstance(history, dict):
    print("Available keys:", history.keys())

    history_df = pd.DataFrame(history)
else:
    history_df = pd.DataFrame(history)

display(history_df)

In [ ]:
print("Available checkpoints:")

for checkpoint in checkpoint_dir.glob("*.pt"):
    print(
        checkpoint.name,
        "| Size:",
        round(checkpoint.stat().st_size / (1024 * 1024), 2),
        "MB"
    )

In [ ]:
print("Available columns:")
print(history_df.columns.tolist())

print("\nFirst epoch record:")
display(history_df.iloc[0])

In [ ]:

display(history_df)

In [ ]:
history_df = pd.DataFrame(history["history"])

display(
    history_df[
        [
            "epoch",
            "train_loss",
            "val_loss",
            "val_pixel_accuracy",
            "val_mean_iou"
        ]
    ].round(4)
)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(
    history_df["epoch"],
    history_df["train_loss"],
    marker="o",
    label="Training loss"
)
axes[0].plot(
    history_df["epoch"],
    history_df["val_loss"],
    marker="o",
    label="Validation loss"
)
axes[0].set_title("Training vs Validation Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(
    history_df["epoch"],
    history_df["val_mean_iou"],
    marker="o"
)
axes[1].set_title("Validation Mean IoU")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Mean IoU")
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
import sys
from pathlib import Path

scripts_dir = dataset_dir / "pipeline_ml" / "scripts"

if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

from common import load_split_csv

print("load_split_csv imported successfully!")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

split_names = ["train", "val", "test"]
sample_details = []

for split_name in split_names:
    samples = load_split_csv(dataset_dir, split_name)

    for sample in samples:
        mask = np.asarray(
            Image.open(sample.mask_path).convert("L")
        )

        positive_pixels = int((mask > 0).sum())

        sample_details.append({
            "split": split_name,
            "sample_id": sample.sample_id,
            "positive_pixels": positive_pixels,
            "positive_fraction": positive_pixels / mask.size
        })

details_df = pd.DataFrame(sample_details)

print("Dataset summary:")
display(
    details_df.groupby("split").agg(
        images=("sample_id", "count"),
        images_without_slick=("positive_pixels", lambda x: (x == 0).sum()),
        median_slick_pixels=("positive_pixels", "median"),
        images_under_1_percent=(
            "positive_fraction", lambda x: (x < 0.01).sum()
        )
    ).round(4)
)

fig, ax = plt.subplots(figsize=(10, 5))

for split_name in split_names:
    values = details_df.loc[
        (details_df["split"] == split_name) &
        (details_df["positive_pixels"] > 0),
        "positive_pixels"
    ]

    ax.hist(values, bins=20, alpha=0.4, label=split_name)

ax.set_xscale("log")
ax.set_xlabel("Slick pixels per image (log scale)")
ax.set_ylabel("Number of images")
ax.set_title("Slick Region Size Distribution")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
batch = next(iter(test_loader))

print("Batch type:", type(batch))

if isinstance(batch, dict):
    print("Dictionary keys:", batch.keys())

    for key, value in batch.items():
        print(
            key,
            "| Type:", type(value),
            "| Shape:", getattr(value, "shape", None)
        )
else:
    print("Batch length:", len(batch))

    for i, value in enumerate(batch):
        print(
            i,
            "| Type:", type(value),
            "| Shape:", getattr(value, "shape", None)
        )

In [ ]:
import torch
import numpy as np

model.eval()
confusion = np.zeros((2, 2), dtype=np.int64)

with torch.no_grad():
    for batch in test_loader:
        images = batch["image"]
        masks = batch["mask"]

        logits = model(images)
        predictions = logits.argmax(dim=1)

        predictions = predictions.cpu().numpy()
        masks = masks.cpu().numpy()

        for pred, target in zip(predictions, masks):
            confusion += np.bincount(
                2 * target.ravel() + pred.ravel(),
                minlength=4
            ).reshape(2, 2)

tn, fp = confusion[0]
fn, tp = confusion[1]

precision = tp / (tp + fp) if tp + fp else 0
recall = tp / (tp + fn) if tp + fn else 0

f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall else 0
)

iou = tp / (tp + fp + fn) if tp + fp + fn else 0
accuracy = (tp + tn) / confusion.sum()

print("FINAL TEST RESULTS — THRESHOLD 0.50")
print("-----------------------------------")
print("Test images:", len(test_samples))
print("Confusion matrix:")
print(confusion)
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"IoU:       {iou:.4f}")
print(f"Accuracy:  {accuracy:.4f}")

In [ ]:
import pandas as pd
import numpy as np
import torch

model.eval()
test_results = []

with torch.no_grad():
    for batch in test_loader:
        images = batch["image"]
        masks = batch["mask"]
        sample_ids = batch["sample_id"]

        predictions = model(images).argmax(dim=1)

        for sample_id, pred, target in zip(
            sample_ids,
            predictions.cpu().numpy(),
            masks.cpu().numpy()
        ):
            tp_i = np.logical_and(pred == 1, target == 1).sum()
            fp_i = np.logical_and(pred == 1, target == 0).sum()
            fn_i = np.logical_and(pred == 0, target == 1).sum()

            iou_i = (
                tp_i / (tp_i + fp_i + fn_i)
                if tp_i + fp_i + fn_i else 0
            )

            test_results.append({
                "sample_id": sample_id,
                "TP": int(tp_i),
                "FP": int(fp_i),
                "FN": int(fn_i),
                "IoU": float(iou_i)
            })

test_results_df = pd.DataFrame(test_results)

display(
    test_results_df.sort_values("IoU")
)

In [ ]:
from pathlib import Path
import sys
import torch
from torch.utils.data import DataLoader

project_root = PROJECT_ROOT
dataset_dir = project_root / "oilspill-detection-dataset_20260912-1b9a62c"

scripts_dir = dataset_dir / "pipeline_ml" / "scripts"
results_dir = dataset_dir / "pipeline_ml" / "results"

if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

from common import load_split_csv, OilSpillSegmentationDataset
from model import TinyUNet

In [ ]:
val_samples = load_split_csv(dataset_dir, "val")

val_dataset = OilSpillSegmentationDataset(val_samples)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False
)

print("Validation images:", len(val_dataset))
print("Validation batches:", len(val_loader))

In [ ]:
import inspect
print(inspect.signature(TinyUNet))


In [ ]:
import json

config_path = scripts_dir / "pipeline_ml_config.json"

with open(config_path, "r") as f:
    config = json.load(f)

print(json.dumps(config, indent=2))

In [ ]:
device = torch.device("cpu")

model = TinyUNet(
    in_channels=3,
    num_classes=2,
    base_ch=32
).to(device)

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state"])
model.eval()

print("Two-class checkpoint loaded successfully!")

In [ ]:
import pandas as pd
import numpy as np
import torch

model.eval()
validation_results = []

with torch.no_grad():
    for batch in val_loader:
        images = batch["image"].to(device)
        masks = batch["mask"].to(device)
        sample_ids = batch["sample_id"]

        probabilities = torch.softmax(model(images), dim=1)[:, 1]
        predictions = probabilities >= 0.50

        for sample_id, pred, target in zip(
            sample_ids,
            predictions.cpu().numpy(),
            masks.cpu().numpy()
        ):
            tp = np.logical_and(pred == 1, target == 1).sum()
            fp = np.logical_and(pred == 1, target == 0).sum()
            fn = np.logical_and(pred == 0, target == 1).sum()

            iou = tp / (tp + fp + fn) if tp + fp + fn else 0

            validation_results.append({
                "sample_id": sample_id,
                "TP": int(tp),
                "FP": int(fp),
                "FN": int(fn),
                "IoU": float(iou)
            })

val_results_df = pd.DataFrame(validation_results)

display(
    val_results_df.sort_values("IoU").reset_index(drop=True)
)

In [ ]:
import matplotlib.pyplot as plt
import torch
import numpy as np

target_ids = [
    "IMG_19_TILE_004",
    "IMG_12_TILE_011",
    "IMG_12_TILE_003",
    "IMG_12_TILE_005"
]

model.eval()

found = {}

with torch.no_grad():
    for batch in val_loader:
        images = batch["image"].to(device)
        masks = batch["mask"]
        sample_ids = batch["sample_id"]

        probs = torch.softmax(model(images), dim=1)[:, 1]

        for i, sample_id in enumerate(sample_ids):
            if sample_id in target_ids:
                found[sample_id] = (
                    images[i].cpu().permute(1, 2, 0).numpy(),
                    masks[i].cpu().numpy(),
                    probs[i].cpu().numpy()
                )

for sample_id in target_ids:
    if sample_id not in found:
        print("Not found:", sample_id)
        continue

    image, actual, probability = found[sample_id]
    predicted = probability >= 0.50

    fig, axes = plt.subplots(1, 4, figsize=(18, 5))

    axes[0].imshow(image)
    axes[0].set_title("Original image")

    axes[1].imshow(actual, cmap="gray")
    axes[1].set_title("Actual oil mask")

    axes[2].imshow(probability, cmap="hot", vmin=0, vmax=1)
    axes[2].set_title("Model probability")

    axes[3].imshow(predicted, cmap="gray")
    axes[3].set_title("Predicted mask (0.50)")

    for ax in axes:
        ax.axis("off")

    plt.suptitle(sample_id)
    plt.tight_layout()
    plt.show()

In [ ]:
import numpy as np
import pandas as pd
import torch

target_ids = [
    "IMG_19_TILE_004",
    "IMG_12_TILE_011",
    "IMG_12_TILE_003",
    "IMG_12_TILE_005"
]

thresholds = [0.30, 0.50, 0.60, 0.70, 0.80]
rows = []

model.eval()

with torch.no_grad():
    for batch in val_loader:
        images = batch["image"].to(device)
        masks = batch["mask"].cpu().numpy()
        sample_ids = batch["sample_id"]

        probabilities = torch.softmax(model(images), dim=1)[:, 1].cpu().numpy()

        for i, sample_id in enumerate(sample_ids):
            if sample_id not in target_ids:
                continue

            target = masks[i] == 1
            prob = probabilities[i]

            for threshold in thresholds:
                pred = prob >= threshold

                tp = np.logical_and(pred, target).sum()
                fp = np.logical_and(pred, ~target).sum()
                fn = np.logical_and(~pred, target).sum()

                iou = tp / (tp + fp + fn) if tp + fp + fn else 0
                precision = tp / (tp + fp) if tp + fp else 0
                recall = tp / (tp + fn) if tp + fn else 0

                rows.append({
                    "sample_id": sample_id,
                    "threshold": threshold,
                    "TP": int(tp),
                    "FP": int(fp),
                    "FN": int(fn),
                    "precision": precision,
                    "recall": recall,
                    "IoU": iou
                })

threshold_comparison = pd.DataFrame(rows)

display(
    threshold_comparison.sort_values(
        ["sample_id", "threshold"]
    ).round(4)
)

In [ ]:
train_script = scripts_dir / "train.py"

print(train_script.read_text(encoding="utf-8"))

In [ ]:
model_script = scripts_dir / "model.py"

print(model_script.read_text(encoding="utf-8"))

In [ ]:
def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--slick-weight", type=float, default=1.0)

In [ ]:
model_script = scripts_dir / "model.py"
print(model_script.read_text(encoding="utf-8"))

In [ ]:
import inspect
from common import metrics_from_confusion, get_num_classes

print("Number of classes:", get_num_classes())
print(inspect.getsource(metrics_from_confusion))

In [ ]:
import inspect
from common import metrics_from_confusion, get_num_classes

print("Number of classes:", get_num_classes())
print("\nMetric calculation:")
print(inspect.getsource(metrics_from_confusion))

In [ ]:
train_samples = load_split_csv(dataset_dir, "train")
train_dataset = OilSpillSegmentationDataset(train_samples)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=0
)

print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Training batches:", len(train_loader))

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class WeightedCEDiceLoss(nn.Module):
    def __init__(self, slick_weight=2.0, dice_weight=1.0):
        super().__init__()

        self.ce = nn.CrossEntropyLoss(
            weight=torch.tensor([1.0, slick_weight])
        )

        self.dice_weight = dice_weight

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(
            logits,
            targets,
            weight=self.ce.weight.to(logits.device)
        )

        probabilities = torch.softmax(logits, dim=1)[:, 1]
        targets_float = (targets == 1).float()

        intersection = (probabilities * targets_float).sum(
            dim=(1, 2)
        )

        denominator = (
            probabilities.sum(dim=(1, 2))
            + targets_float.sum(dim=(1, 2))
        )

        dice = (2 * intersection + 1e-6) / (denominator + 1e-6)

        dice_loss = 1 - dice.mean()

        return ce_loss + self.dice_weight * dice_loss

### Weighted CE + Dice experiment setup

The following cells define and train an additional segmentation experiment. Preserve its original architecture/loss logic; execution may create or overwrite files in the named run directory. No training was started during cleanup.

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import time

from pathlib import Path
from torch.utils.data import DataLoader
from common import set_seed, update_confusion, metrics_from_confusion
from model import TinyUNet

set_seed(42)

device = torch.device("cpu")

experiment_dir = results_dir / "cpu_weighted_ce_dice_10epochs"
checkpoints_dir = experiment_dir / "checkpoints"
checkpoints_dir.mkdir(parents=True, exist_ok=True)

print("Experiment folder:", experiment_dir)
print("Device:", device)

In [ ]:
class WeightedCEDiceLoss(nn.Module):
    def __init__(self, slick_weight=2.0, dice_weight=1.0):
        super().__init__()
        self.slick_weight = slick_weight
        self.dice_weight = dice_weight

    def forward(self, logits, targets):
        weights = torch.tensor(
            [1.0, self.slick_weight],
            device=logits.device
        )

        ce_loss = F.cross_entropy(
            logits,
            targets,
            weight=weights
        )

        probabilities = torch.softmax(logits, dim=1)[:, 1]
        targets_float = (targets == 1).float()

        intersection = (probabilities * targets_float).sum(dim=(1, 2))

        denominator = (
            probabilities.sum(dim=(1, 2))
            + targets_float.sum(dim=(1, 2))
        )

        dice_score = (
            2 * intersection + 1e-6
        ) / (
            denominator + 1e-6
        )

        dice_loss = 1 - dice_score.mean()

        return ce_loss + self.dice_weight * dice_loss


criterion = WeightedCEDiceLoss(
    slick_weight=2.0,
    dice_weight=1.0
)

print("Weighted CE + Dice loss ready.")

In [ ]:
model = TinyUNet(
    in_channels=3,
    num_classes=2,
    base_ch=32
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Model initialized.")
print("Trainable parameters:",
      sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
import json
import time

def evaluate_model(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    total_samples = 0
    conf = np.zeros((2, 2), dtype=np.int64)

    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(device)
            masks = batch["mask"].to(device)

            logits = model(images)
            loss = criterion(logits, masks)

            total_loss += loss.item() * images.size(0)
            total_samples += images.size(0)

            predictions = torch.argmax(logits, dim=1)

            update_confusion(
                conf,
                masks.cpu().numpy(),
                predictions.cpu().numpy(),
                num_classes=2
            )

    metrics = metrics_from_confusion(conf)
    average_loss = total_loss / total_samples

    return average_loss, metrics, conf


def train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    epochs=10
):
    best_iou = -1.0
    history = []

    for epoch in range(1, epochs + 1):
        start_time = time.time()
        model.train()

        total_loss = 0.0
        total_samples = 0

        for batch in train_loader:
            images = batch["image"].to(device)
            masks = batch["mask"].to(device)

            optimizer.zero_grad()

            logits = model(images)
            loss = criterion(logits, masks)

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * images.size(0)
            total_samples += images.size(0)

        train_loss = total_loss / total_samples

        val_loss, val_metrics, val_conf = evaluate_model(
            model, val_loader, criterion, device
        )

        val_iou = val_metrics["mean_iou"]
        elapsed = time.time() - start_time

        epoch_result = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_iou": val_iou,
            "val_precision": val_metrics["precision"],
            "val_recall": val_metrics["recall"],
            "val_f1": val_metrics["f1_score"],
            "seconds": elapsed
        }

        history.append(epoch_result)

        checkpoint = {
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "val_metrics": val_metrics,
            "config": {
                "slick_weight": 2.0,
                "dice_weight": 1.0,
                "learning_rate": 0.001,
                "seed": 42
            }
        }

        torch.save(checkpoint, checkpoints_dir / "last.pt")

        if val_iou > best_iou:
            best_iou = val_iou
            torch.save(checkpoint, checkpoints_dir / "best.pt")

        with open(experiment_dir / "history.json", "w") as f:
            json.dump(history, f, indent=4)

        print(
            f"Epoch {epoch:02d}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val IoU: {val_iou:.4f} | "
            f"Precision: {val_metrics['precision']:.4f} | "
            f"Recall: {val_metrics['recall']:.4f} | "
            f"F1: {val_metrics['f1_score']:.4f} | "
            f"Time: {elapsed / 60:.1f} min"
        )

    print("\nTraining completed.")
    print(f"Best validation IoU: {best_iou:.4f}")

    return history


history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=10
)

In [ ]:
import json
import time

def evaluate_model(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    total_samples = 0
    conf = np.zeros((2, 2), dtype=np.int64)

    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(device)
            masks = batch["mask"].to(device)

            logits = model(images)
            loss = criterion(logits, masks)

            total_loss += loss.item() * images.size(0)
            total_samples += images.size(0)

            predictions = torch.argmax(logits, dim=1)

            update_confusion(
                conf,
                masks.cpu().numpy(),
                predictions.cpu().numpy(),
                num_classes=2
            )

    metrics = metrics_from_confusion(conf)
    average_loss = total_loss / total_samples

    return average_loss, metrics, conf


def train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    epochs=10
):
    best_iou = -1.0
    history = []

    for epoch in range(1, epochs + 1):
        start_time = time.time()
        model.train()

        total_loss = 0.0
        total_samples = 0

        for batch in train_loader:
            images = batch["image"].to(device)
            masks = batch["mask"].to(device)

            optimizer.zero_grad()

            logits = model(images)
            loss = criterion(logits, masks)

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * images.size(0)
            total_samples += images.size(0)

        train_loss = total_loss / total_samples

        val_loss, val_metrics, val_conf = evaluate_model(
            model, val_loader, criterion, device
        )

        val_iou = val_metrics["mean_iou"]
        elapsed = time.time() - start_time

        epoch_result = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_iou": val_iou,
            "val_precision": val_metrics["precision"],
            "val_recall": val_metrics["recall"],
            "val_f1": val_metrics["f1_score"],
            "seconds": elapsed
        }

        history.append(epoch_result)

        checkpoint = {
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "val_metrics": val_metrics,
            "config": {
                "slick_weight": 2.0,
                "dice_weight": 1.0,
                "learning_rate": 0.001,
                "seed": 42
            }
        }

        torch.save(checkpoint, checkpoints_dir / "last.pt")

        if val_iou > best_iou:
            best_iou = val_iou
            torch.save(checkpoint, checkpoints_dir / "best.pt")

        with open(experiment_dir / "history.json", "w") as f:
            json.dump(history, f, indent=4)

        print(
            f"Epoch {epoch:02d}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val IoU: {val_iou:.4f} | "
            f"Precision: {val_metrics['precision']:.4f} | "
            f"Recall: {val_metrics['recall']:.4f} | "
            f"F1: {val_metrics['f1_score']:.4f} | "
            f"Time: {elapsed / 60:.1f} min"
        )

    print("\nTraining completed.")
    print(f"Best validation IoU: {best_iou:.4f}")

    return history


history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=10
)

In [ ]:
checkpoint_path = checkpoints_dir / "best.pt"

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

print("Best epoch:", checkpoint["epoch"])
print("Validation metrics:")

for key, value in checkpoint["val_metrics"].items():
    print(key, ":", value)

## 10. Checkpoint comparisons, threshold analysis and test evaluation

Compare the retained checkpoints and inspect threshold sweeps, per-image predictions and qualitative examples. Threshold/model selection must be based on validation data; test evaluation is final-only. Confirm the provenance of any fixed test threshold before interpreting reported test metrics.

In [ ]:
import torch
import numpy as np
import pandas as pd

# Checkpoint locations
old_checkpoint_path = (
    results_dir / "cpu_dice_ce_10epochs" /
    "checkpoints" / "best.pt"
)

new_checkpoint_path = (
    results_dir / "cpu_weighted_ce_dice_10epochs" /
    "checkpoints" / "best.pt"
)

# Load a checkpoint into the same model architecture
def load_model_for_comparison(checkpoint_path):
    comparison_model = TinyUNet(
        in_channels=3,
        num_classes=2,
        base_ch=32
    ).to(device)

    saved = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=False
    )

    comparison_model.load_state_dict(saved["model_state"])
    comparison_model.eval()

    return comparison_model, saved["epoch"]


old_model, old_epoch = load_model_for_comparison(
    old_checkpoint_path
)

new_model, new_epoch = load_model_for_comparison(
    new_checkpoint_path
)

print("Original CE + Dice best epoch:", old_epoch)
print("Weighted CE + Dice best epoch:", new_epoch)


# Evaluate both models image by image
def compare_models_per_image(model, loader, model_name, threshold=0.50):
    rows = []
    model.eval()

    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(device)
            masks = batch["mask"].cpu().numpy()
            sample_ids = batch["sample_id"]

            logits = model(images)
            probabilities = torch.softmax(logits, dim=1)[:, 1]
            predictions = (
                probabilities >= threshold
            ).cpu().numpy()

            for i, sample_id in enumerate(sample_ids):
                actual = masks[i] == 1
                predicted = predictions[i]

                tp = int(np.logical_and(actual, predicted).sum())
                fp = int(np.logical_and(~actual, predicted).sum())
                fn = int(np.logical_and(actual, ~predicted).sum())

                iou = tp / (tp + fp + fn + 1e-9)
                precision = tp / (tp + fp + 1e-9)
                recall = tp / (tp + fn + 1e-9)

                rows.append({
                    "sample_id": sample_id,
                    "model": model_name,
                    "TP": tp,
                    "FP": fp,
                    "FN": fn,
                    "IoU": iou,
                    "Precision": precision,
                    "Recall": recall
                })

    return pd.DataFrame(rows)


old_results = compare_models_per_image(
    old_model,
    val_loader,
    "Original CE + Dice"
)

new_results = compare_models_per_image(
    new_model,
    val_loader,
    "Weighted CE + Dice"
)

comparison = old_results.merge(
    new_results,
    on="sample_id",
    suffixes=("_original", "_weighted")
)

comparison["IoU_difference"] = (
    comparison["IoU_weighted"] - comparison["IoU_original"]
)

comparison = comparison.sort_values("IoU_difference")

display(comparison)

# Overall comparison using pooled pixel counts
def pooled_metrics(results, model_name):
    tp = results["TP"].sum()
    fp = results["FP"].sum()
    fn = results["FN"].sum()

    return {
        "Model": model_name,
        "Precision": tp / (tp + fp + 1e-9),
        "Recall": tp / (tp + fn + 1e-9),
        "F1": 2 * tp / (2 * tp + fp + fn + 1e-9),
        "IoU": tp / (tp + fp + fn + 1e-9)
    }


summary = pd.DataFrame([
    pooled_metrics(old_results, "Original CE + Dice"),
    pooled_metrics(new_results, "Weighted CE + Dice")
])

print("\nOverall pooled validation comparison:")
display(summary)

# Save the comparison
comparison.to_csv(
    experiment_dir / "validation_per_image_comparison.csv",
    index=False
)

summary.to_csv(
    experiment_dir / "validation_model_summary.csv",
    index=False
)

print("Comparison files saved in:", experiment_dir)

In [ ]:
thresholds = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]

def evaluate_threshold(model, loader, threshold):
    model.eval()

    conf = np.zeros((2, 2), dtype=np.int64)

    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(device)
            masks = batch["mask"].cpu().numpy()

            logits = model(images)
            probabilities = torch.softmax(logits, dim=1)[:, 1]

            predictions = (
                probabilities >= threshold
            ).cpu().numpy().astype(np.int64)

            update_confusion(
                conf,
                masks,
                predictions,
                num_classes=2
            )

    return metrics_from_confusion(conf)


threshold_results = []

for threshold in thresholds:
    metrics = evaluate_threshold(
        new_model,
        val_loader,
        threshold
    )

    threshold_results.append({
        "Threshold": threshold,
        "Precision": metrics["precision"],
        "Recall": metrics["recall"],
        "F1": metrics["f1_score"],
        "Slick_IoU": metrics["mean_iou"]
    })

threshold_df = pd.DataFrame(threshold_results)

display(threshold_df)

In [ ]:
original_threshold_results = []

for threshold in thresholds:
    metrics = evaluate_threshold(
        old_model,
        val_loader,
        threshold
    )

    original_threshold_results.append({
        "Threshold": threshold,
        "Precision": metrics["precision"],
        "Recall": metrics["recall"],
        "F1": metrics["f1_score"],
        "Slick_IoU": metrics["mean_iou"]
    })

original_threshold_df = pd.DataFrame(
    original_threshold_results
)

display(original_threshold_df)

In [ ]:
final_checkpoint_path = (
    results_dir
    / "cpu_weighted_ce_dice_10epochs"
    / "checkpoints"
    / "best.pt"
)

final_checkpoint = torch.load(
    final_checkpoint_path,
    map_location=device,
    weights_only=False
)

print("Checkpoint epoch:", final_checkpoint["epoch"])
print("Saved configuration:", final_checkpoint["config"])
print("Validation IoU:", final_checkpoint["val_metrics"]["mean_iou"])
print("Checkpoint exists:", final_checkpoint_path.exists())

In [ ]:
# Load the selected weighted checkpoint
final_model = TinyUNet(
    in_channels=3,
    num_classes=2,
    base_ch=32
).to(device)

final_model.load_state_dict(final_checkpoint["model_state"])
final_model.eval()

# Load the untouched test split
test_samples = load_split_csv(dataset_dir, "test")
test_dataset = OilSpillSegmentationDataset(test_samples)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0
)

# Evaluate at the validation-selected threshold
TEST_THRESHOLD = 0.90

test_metrics = evaluate_threshold(
    final_model,
    test_loader,
    TEST_THRESHOLD
)

print("FINAL TEST RESULTS")
print("------------------")
print("Test images:", len(test_dataset))
print("Threshold:", TEST_THRESHOLD)
print("Precision:", round(test_metrics["precision"], 4))
print("Recall:", round(test_metrics["recall"], 4))
print("F1-score:", round(test_metrics["f1_score"], 4))
print("Slick IoU:", round(test_metrics["mean_iou"], 4))
print("Pixel accuracy:", round(test_metrics["pixel_accuracy"], 4))

In [ ]:
# Load the original CE + Dice checkpoint
original_checkpoint_path = (
    results_dir
    / "cpu_dice_ce_10epochs"
    / "checkpoints"
    / "best.pt"
)

original_checkpoint = torch.load(
    original_checkpoint_path,
    map_location=device,
    weights_only=False
)

original_model = TinyUNet(
    in_channels=3,
    num_classes=2,
    base_ch=32
).to(device)

original_model.load_state_dict(
    original_checkpoint["model_state"]
)

original_model.eval()

# Evaluate at the threshold selected using validation data
original_test_metrics = evaluate_threshold(
    original_model,
    test_loader,
    0.70
)

print("ORIGINAL MODEL — FINAL TEST")
print("---------------------------")
print("Threshold: 0.70")
print("Precision:", round(original_test_metrics["precision"], 4))
print("Recall:", round(original_test_metrics["recall"], 4))
print("F1-score:", round(original_test_metrics["f1_score"], 4))
print("Slick IoU:", round(original_test_metrics["mean_iou"], 4))
print("Pixel accuracy:", round(original_test_metrics["pixel_accuracy"], 4))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

final_model.eval()

TEST_THRESHOLD = 0.90

test_predictions = []

with torch.no_grad():
    for batch in test_loader:
        images = batch["image"].to(device)
        masks = batch["mask"].cpu().numpy()

        logits = final_model(images)
        probabilities = torch.softmax(logits, dim=1)[:, 1]
        predictions = (
            probabilities >= TEST_THRESHOLD
        ).cpu().numpy().astype(np.uint8)

        for i in range(len(predictions)):
            test_predictions.append({
                "sample_id": batch["sample_id"][i],
                "image": images[i].cpu().permute(1, 2, 0).numpy(),
                "ground_truth": masks[i],
                "prediction": predictions[i],
                "probability": probabilities[i].cpu().numpy()
            })

print("Test predictions generated:", len(test_predictions))

In [ ]:
for sample in test_predictions[:3]:
    print(
        sample["sample_id"],
        "| Ground-truth slick pixels:",
        int(sample["ground_truth"].sum()),
        "| Predicted slick pixels:",
        int(sample["prediction"].sum())
    )

In [ ]:
def show_prediction(sample):
    image = sample["image"]
    gt = sample["ground_truth"]
    pred = sample["prediction"]
    prob = sample["probability"]

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    axes[0].imshow(image)
    axes[0].set_title("Original image")

    axes[1].imshow(gt, cmap="gray", vmin=0, vmax=1)
    axes[1].set_title("Ground truth")

    axes[2].imshow(pred, cmap="gray", vmin=0, vmax=1)
    axes[2].set_title("Model prediction")

    axes[3].imshow(image)
    axes[3].imshow(pred, cmap="Reds", alpha=0.45)
    axes[3].set_title("Prediction overlay")

    for ax in axes:
        ax.axis("off")

    fig.suptitle(sample["sample_id"], fontsize=14)
    plt.tight_layout()
    plt.show()


for sample in test_predictions[:3]:
    show_prediction(sample)

In [ ]:
def calculate_image_iou(gt, pred):
    intersection = np.logical_and(gt == 1, pred == 1).sum()
    union = np.logical_or(gt == 1, pred == 1).sum()

    if union == 0:
        return 1.0

    return intersection / union


for sample in test_predictions:
    sample["iou"] = calculate_image_iou(
        sample["ground_truth"],
        sample["prediction"]
    )

test_predictions_sorted = sorted(
    test_predictions,
    key=lambda x: x["iou"]
)

print("WEAKEST 5 TEST PREDICTIONS")
for sample in test_predictions_sorted[:5]:
    print(sample["sample_id"], "IoU:", round(sample["iou"], 4))

print("\nSTRONGEST 5 TEST PREDICTIONS")
for sample in test_predictions_sorted[-5:][::-1]:
    print(sample["sample_id"], "IoU:", round(sample["iou"], 4))

In [ ]:
print("WEAKEST TEST EXAMPLES")
for sample in test_predictions_sorted[:3]:
    show_prediction(sample)

print("STRONGEST TEST EXAMPLES")
for sample in test_predictions_sorted[-3:][::-1]:
    show_prediction(sample)

In [ ]:
worst_samples = test_predictions_sorted[:2]

for sample in worst_samples:
    print("Sample:", sample["sample_id"])
    print("Ground-truth pixels:", int(sample["ground_truth"].sum()))
    print("Predicted pixels:", int(sample["prediction"].sum()))
    print("IoU:", round(sample["iou"], 4))

    show_prediction(sample)

In [ ]:
# Inspect probability maps for the two worst test examples

for sample in test_predictions_sorted[:2]:

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    axes[0].imshow(sample["image"])
    axes[0].set_title("Original image")

    axes[1].imshow(sample["ground_truth"], cmap="gray")
    axes[1].set_title("Ground truth")

    im = axes[2].imshow(
        sample["probability"],
        cmap="hot",
        vmin=0,
        vmax=1
    )
    axes[2].set_title("Oil-spill probability")

    plt.colorbar(im, ax=axes[2], fraction=0.046)

    fig.suptitle(sample["sample_id"])
    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

    print(
        "Maximum predicted probability:",
        round(float(sample["probability"].max()), 4)
    )

In [ ]:
for sample in test_predictions_sorted[:2]:

    gt = sample["ground_truth"] == 1
    prob = sample["probability"]

    inside = prob[gt]
    outside = prob[~gt]

    print("\nSample:", sample["sample_id"])
    print("Maximum probability:", round(float(prob.max()), 4))

    print(
        "Mean probability inside actual slick:",
        round(float(inside.mean()), 4) if inside.size else "N/A"
    )

    print(
        "Maximum probability inside actual slick:",
        round(float(inside.max()), 4) if inside.size else "N/A"
    )

    print(
        "Mean probability outside actual slick:",
        round(float(outside.mean()), 4) if outside.size else "N/A"
    )

    print(
        "Pixels above 0.90 inside actual slick:",
        int((inside >= 0.90).sum())
    )

    print(
        "Pixels above 0.90 outside actual slick:",
        int((outside >= 0.90).sum())
    )

## 11. Consolidated results, limitations and next steps

Use the saved validation histories/checkpoints and final test outputs as the source of reported metrics. Keep global pixel metrics separate from averages over individual images. The notebook contains exploratory and historical experiments as well as final-evaluation cells; do not treat every experiment as a selected model.

Before claiming a final model or reproducible run:
- Confirm the configured dataset, split files, SAFE archive, scripts and checkpoints exist.
- Re-run the intended workflow in order in a configured environment.
- Verify checkpoint/CSV/history metric agreement.
- Confirm all chosen thresholds were selected on validation data only.
- Record package versions, hardware, configuration and run identifiers.